# 31556268
XOR as a Regression problem:
Output Similiar to random, all around 0.5.

Loss, Hidden layer activation  
Adam instead of sgd is also suggested  
It Also needs 2000 epochs

******* Confirm ground truth *******

In [ ]:
from keras.models import Sequential
from keras.layers import Dense,Activation # Was from keras.layers.core import Dense,Activation
from keras.optimizers import SGD
import numpy as np
import time

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

model = Sequential()
model.add(Dense(4, input_dim=2,kernel_initializer="glorot_uniform")) # Was model.add(Dense(input_dim=2,output_dim=4,init="glorot_uniform")) -- Keras < 2.0
model.add(Activation("sigmoid"))
# + model.add(Activation("relu"))
model.add(Dense(1,kernel_initializer="glorot_uniform")) # model.add(Dense(input_dim=4,output_dim=1,init="glorot_uniform")) -- Keras < 2.0
model.add(Activation("sigmoid"))
sgd = SGD(0.05, decay=1e-6, momentum=0.11, nesterov=True) # Was sgd = SGD(l2=0.0,lr=0.05, decay=1e-6, momentum=0.11, nesterov=True)
model.compile(loss='mean_absolute_error', optimizer=sgd,  metrics=['accuracy'])
# + model.compile(loss='mse', optimizer=sgd,  metrics=['accuracy'])
list1 = [1,1]
label1 = [0]
list2 = [1,0]
label2 = [1]
list3 = [0,0]
label3 = [0]
list4 = [0,1]
label4 = [1] 
train_data = np.array((list1,list2,list3,list4))
label = np.array((label1,label2,label3,label4))

model.fit(train_data,label,epochs = 1000,batch_size = 4,verbose = 1,shuffle=True) # Was model.fit(train_data,label,nb_epoch = 1000,batch_size = 4,verbose = 1,shuffle=True,show_accuracy = True)
# list_test = [0,1]
# test = np.array((list_test,list1))
# ------------------------- test = np.array((list1,list2, list3, list4))
# ------------------------- classes = model.predict(test)
# ------------------------- print (classes) # Was without Parentheses


modeltype = "Regression" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "[0, 1]" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 1 # Number of classes for classification and target shape for regression
input_shape = (2,) # Just input shape without batch size or channels

HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument

end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 31627380
Classification problem, Titanic Dataset

All loss and accuracy Zero, keras itself mentions softmax in a vague way  
Incorrect preprocessing leading to weired results even after normalization  

Output activation and loss

In [ ]:
from keras.models import Sequential
#from keras.layers.core import Dense, Activation, Dropout
#from keras.layers.advanced_activations import PReLU, LeakyReLU
#from keras.layers.recurrent import SimpleRNN, SimpleDeepRNN
#from keras.layers.embeddings import Embedding
#from keras.layers.recurrent import LSTM, GRU
from keras.layers import Dense, Activation, Dropout
import pandas as pd
import numpy as np 
from sklearn.preprocessing import LabelEncoder # Was from sklearn import preprocessing
import time

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

np.random.seed(1919)
data_folder = "./Datasets/titanic/"
batch_size = 4
epochs = 10

train  = pd.read_csv(data_folder+'train.csv', index_col=0)
test  = pd.read_csv(data_folder+'test.csv', index_col=0)
Y = train.Survived
train.drop('Survived', axis=1, inplace=True)
columns = train.columns
test_ind = test.index

train['Age'] = train['Age'].fillna(train['Age'].mean())
test['Age'] = test['Age'].fillna(test['Age'].mean())
train['Fare'] = train['Fare'].fillna(train['Fare'].mean())
test['Fare'] = test['Fare'].fillna(test['Fare'].mean())

category_index = [0,1,2,4,5,6,8,9]
for i in category_index:
    train[columns[i]] = train[columns[i]].fillna('missing')
    test[columns[i]] = test[columns[i]].fillna('missing')

train = np.array(train)
test = np.array(test)

for i in category_index:
    lbl = LabelEncoder() # Was lbl = preprocessing.LabelEncoder()
    lbl.fit(list(train[:,i]) + list(test[:,i]))
    train[:,i] = lbl.transform(train[:,i])
    test[:,i] = lbl.transform(test[:,i])

train = train.astype(np.float32)
test = test.astype(np.float32)

##### Added To check NORMALIZATION
# ************************* print(np.mean(train, axis = 0))
# ************************* print(np.var(train, axis = 0))
# ************************* print("", np.min(train, axis = 0))
# ************************* print("", np.max(train, axis = 0))

model = Sequential()
model.add(Dense(512, input_dim=len(columns))) # Was model.add(Dense(len(columns), 512)) very old keras
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(1)) # Was model.add(Dense(512, 1))
model.add(Activation('softmax'))
# + model.add(Activation('sigmoid'))
model.compile(loss='categorical_crossentropy', optimizer="adam")
# + model.compile(loss='binary_crossentropy', optimizer="adam", metrics=['accuracy']) # Added Accuracy
model.fit(train, Y, epochs=epochs, batch_size=batch_size, validation_split=0.20) # Was model.fit(train, Y, nb_epoch=nb_epoch, batch_size=batch_size, validation_split=0.20)
# preds = model.predict(test,batch_size=batch_size)
# pred_arr = []
# for pred in preds:
    # pred_arr.append(pred[0])
# preds = pd.DataFrame({"PassengerId": test_ind, "Survived": pred_arr})
# preds = preds.set_index('PassengerId')
# preds.to_csv(out_folder+'test.csv')


modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (10,) # Just input shape without batch size or channels

HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument

end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 31880720
Binary classifier:
Not MNIST but seems like a random dataset

Very low test accuracy, Keras itself warns about softmax


In [ ]:
from keras.models import Sequential # Was missing
from keras.layers import Dense,Activation, Dropout # Was missing
import numpy # Was missing
import keras.optimizers # Was missing
from keras.datasets import mnist # USELESS
import time

(x_tr, y_tr), (x_te, y_te) = mnist.load_data() # USELESS
print (x_tr.shape) # Was print x_tr.shape
X_train = numpy.array([[1] * 128] * (10 ** 4) + [[0] * 128] * (10 ** 4)) # It's 128 columns of 1 repeated 10000 times plus same for zero
X_test = numpy.array([[1] * 128] * (10 ** 2) + [[0] * 128] * (10 ** 2))

Y_train = numpy.array([True] * (10 ** 4) + [False] * (10 ** 4)) # It's 10000 true followed by 10000 False
Y_test = numpy.array([True] * (10 ** 2) + [False] * (10 ** 2)) # It's 100 True followed by 100 False

X_train = X_train.astype("float32")
X_test = X_test.astype("float32")

Y_train = Y_train.astype("bool")
Y_test = Y_test.astype("bool")
print (X_train.shape)

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

batch_size = 4 # Was missing
epochs = 10 # Was missing

model = Sequential()
model.add(Dense(50, input_dim = 128)) # Was model.add(Dense(128, 50))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Dense(50)) # Was model.add(Dense(50, 50))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Dense(1)) # Was model.add(Dense(50, 1))
model.add(Activation('softmax'))
# + model.add(Activation('sigmoid'))
rms = keras.optimizers.RMSprop() # Was RMSprp()
model.compile(loss='binary_crossentropy', optimizer=rms, metrics=['accuracy']) # Added Accuracy
model.fit(X_train, Y_train, batch_size=batch_size, epochs=epochs, verbose=2, validation_data=(X_test, Y_test))
# Was model.fit(X_train, Y_train, batch_size=batch_size, epochs=epochs, show_accuracy=True, verbose=2, validation_data=(X_test, Y_test))

#score = model.evaluate(X_test, Y_test, verbose=0)
#print(model.metrics_names) # Added
#print('Test score:', score[0])
#print('Test accuracy:', score[1])

modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (128,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")


# ******************** We have not trained the model yet 
# score = model.evaluate(X_test, Y_test, verbose=0)
#print(model.metrics_names) # Added
#print('Test score:', score[0])
# print('Test accuracy:', score[1])

# 33969059
Regression

Random Dataset of points (defined weights with added noise)

Very High loss and all predicted all rows the same. Keras warns about softmax

Zero accuracy: Because accuracy is not a proper metric for regression

Standardized data including target variables, and loss dropped significantly


In [ ]:
from keras.models import Sequential # Removed Graph
from keras.layers import Dense, Dropout, Activation # Was from keras.layers.core import Dense, Dropout, Activation
from keras.optimizers import SGD
import numpy as np
import sys # Remove theano
from sklearn.preprocessing import StandardScaler # Added To normalize
import time

NUM_TRAIN = 100000
NUM_TEST = 10000
INDIM = 3
EPOCHS = 20 # Added
mn = 1

def myrand(a, b) :
    return (b)*(np.random.random_sample()-0.5)+a

def get_data(count, ws, xno, bounds=100, rweight=0.0) :
    xt = np.random.rand(count, len(ws))
    xt = np.multiply(bounds, xt)
    yt = np.random.rand(count, 1)
    ws = np.array(ws, dtype=float) # Was np.float instead of float
    xno = np.array([float(xno) + rweight*myrand(-mn, mn) for x in xt], dtype=float) # Was np.float instead of float
    yt = np.dot(xt, ws)
    yt = np.add(yt, xno)

    return (xt, yt)

if __name__ == '__main__' :
    if len(sys.argv) > 3 : # WAS if len(sys.argv) > 1 but I was not giving Arguments and was already 3 
        print (sys.argv[1])
        EPOCHS = int(sys.argv[1])
        XNO = float(sys.argv[2])
        WS = [float(x) for x in sys.argv[3:]]
        mx = max([abs(x) for x in (WS+[XNO])])
        mn = min([abs(x) for x in (WS+[XNO])])
        mn = min(1, mn)
        WS = [float(x)/mx for x in WS]
        XNO = float(XNO)/mx
        INDIM = len(WS)
    else :
        INDIM = 3
        WS = [2.0, 1.0, 0.5]
        XNO = 2.2
  

    X_test, y_test = get_data(10000, WS, XNO, 10000, rweight=0.4)
    print ("X_test shape: ", X_test.shape, "y_test shape: ", y_test.shape)
    X_train, y_train = get_data(100000, WS, XNO, 10000)
    print ("X_train shape: ", X_train.shape, "y_train shape: ", y_train.shape)

    # print("########### first X_train: ", X_train[10])
    # print("########### first y_train: ", y_train[10])
    # print("########### first y_test: ", y_test[10])
    # print ("Train Min and Max: ", np.min(y_train), np.max(y_test))

    # ********************* Added to normalize
    # sc_x = StandardScaler()
    # sc_y = StandardScaler()
    # X_train = sc_x.fit_transform(X_train)
    # X_test = sc_x.transform(X_test)
    # y_train = y_train.reshape(-1,1)
    # y_test = y_test.reshape(-1,1)
    # print ("y_train shape becomes: ", y_train.shape, "y_test shape becomes: ", y_test.shape)
    # y_train = sc_y.fit_transform(y_train) # Added to normalize
    #c y_test = sc_y.transform(y_test) # Added to normalize
    # print ("X_test shape becomes: ", X_test.shape, "y_test shape becomes: ", y_test.shape)
    # print ("X_train shape becomes: ", X_train.shape, "y_train shape becomes: ", y_train.shape)
    # print ("Train Min and Max: ", np.min(y_train), np.max(y_test)) 




    # *************************** Added To check NORMALIZATION
    # print("After normalization: ", np.mean(X_train, axis = 0))
    # print(np.var(X_train, axis = 0))
    # print(np.min(X_train, axis = 0))
    # print(np.max(X_train, axis = 0))

    

    start_time = time.perf_counter()
    %run kerassource/HomaTect.ipynb

    

    model = Sequential()
    model.add(Dense(INDIM, input_dim=INDIM, activation='tanh', kernel_initializer="random_uniform")) # WAS model.add(Dense(INDIM, input_dim=INDIM, init='uniform', activation='tanh'))
    model.add(Dropout(0.5))
    model.add(Dense(2, activation='tanh', kernel_initializer="random_uniform")) # WAS model.add(Dense(2, init='uniform', activation='tanh'))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='softmax', kernel_initializer="random_uniform"))
    # + model.add(Dense(1, activation='linear', kernel_initializer="random_uniform")) # WAS model.add(Dense(1, init='uniform', activation='softmax'))
    sgd = SGD(0.1, decay=1e-6, momentum=0.9, nesterov=True) # WAS sgd = SGD(lr=0.1, decay=1e-6, momentum=0.9, nesterov=True)
    model.compile(loss='mean_squared_error', optimizer=sgd, metrics=['accuracy']) # Added accuracy from fit

    ## print("########### X_train: ", X_train[10])
    ## print("########### y_train: ", y_train[10])
    ## print("########### y_test: ", y_test[10])


    modeltype = "regression" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
    target = "[-n,n]" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
    output_shape = 1 # Number of classes for classification and target shape for regression
    input_shape = (3,) # Just input shape without batch size or channels
    HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
    end_time = time.perf_counter()
    print(f"It took {end_time - start_time} seconds")

    
        
    model.fit(X_train, y_train, shuffle="batch", epochs=EPOCHS) # WAS model.fit(X_train, y_train, shuffle="batch", show_accuracy=True, nb_epoch=EPOCHS)
   
    # ******************** Model isnt trained yet
    # score, acc = model.evaluate(X_test, y_test, batch_size=16) # WAS score, acc = model.evaluate(X_test, y_test, batch_size=16, show_accuracy=True)
    # print (score) # Parentheses
    # print (acc) # Parentheses
    # predictions = sc_y.inverse_transform(model.predict(sc_x.transform([X_train[10], X_train[11]]))) # Added
    # print ("%s --> %s" % (str(X_train[10]), str(predictions[0]))) # Added
    # print ("%s --> %s" % (str(X_train[11]), str(predictions[1]))) # Added
    # predict_data = np.random.rand(100*100, INDIM)
    # - predictions = model.predict(predict_data)
    # predictions = sc_y.inverse_transform(model.predict(sc_x.transform(predict_data))) # Added

    # for x in range(len(predict_data)) :
        # print ("%s --> %s" % (str(predict_data[x]), str(predictions[x]))) # Parentheses

# 34311586
XOR problem

Softmax, keras warns too

also learing rate is too ? and batch_size needs to be defined, train on more epochs

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation # Was from keras.layers.core import Dense, Dropout, Activation
from keras.optimizers import SGD
import numpy
import time

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

X = numpy.array([[1., 1.], [0., 0.], [1., 0.], [0., 1.], [1., 1.], [0., 0.]])
y = numpy.array([[0.], [0.], [1.], [1.], [0.], [0.]])
model = Sequential()
model.add(Dense(2, input_dim=2, activation='sigmoid', kernel_initializer="random_uniform")) # Was model.add(Dense(2, input_dim=2, init='uniform', activation='sigmoid'))
model.add(Dense(3, activation='sigmoid', kernel_initializer="random_uniform")) # Was model.add(Dense(3, init='uniform', activation='sigmoid'))
model.add(Dense(1, activation='softmax', kernel_initializer="random_uniform"))
# + model.add(Dense(1, activation='sigmoid', kernel_initializer="random_uniform")) # Was model.add(Dense(1, init='uniform', activation='softmax'))
sgd = SGD(0.001, decay=1e-6, momentum=0.9, nesterov=True) # Was sgd = SGD(lr=0.001, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(loss='mean_squared_error', optimizer=sgd)
model.fit(X, y, epochs=1000) # Was model.fit(X, y, nb_epoch=20)

modeltype = "regression" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "[0,1]" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 1 # Number of classes for classification and target shape for regression
input_shape = (2,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# *********************** Model is not trained yet 
# print()
# score = model.evaluate(X, y)
# print()
# print(score)
# print(model.predict(numpy.array([[1, 0]])))
# print(model.predict(numpy.array([[0, 0]])))

# 34673164

Binary classification so loss and activation

Dataset not available so probably other factors matter too

In [ ]:
from keras.models import Sequential # Added
from keras.layers import Activation, Dense, Dropout # Added
from keras.optimizers import SGD # Added
import time

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

model = Sequential()
model.add(Dense(64, input_dim=14, kernel_initializer="random_uniform")) # Was model.add(Dense(64, input_dim=14, init='uniform'))
model.add(Activation('tanh'))
model.add(Dropout(0.5))
model.add(Dense(64, kernel_initializer="random_uniform")) # Was model.add(Dense(64, init='uniform'))
model.add(Activation('tanh'))
model.add(Dropout(0.5))
model.add(Dense(2, kernel_initializer="random_uniform")) # Was model.add(Dense(2, init='uniform'))
# + model.add(Dense(1, kernel_initializer="random_uniform"))
model.add(Activation('softmax'))
# + model.add(Activation('sigmoid'))

sgd = SGD(0.1, decay=1e-6, momentum=0.9, nesterov=True) # Was sgd = SGD(lr=0.1, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(loss='mean_squared_error', optimizer=sgd)
# + model.compile(loss='binary_crossentropy', optimizer=sgd)

# Dataset not defined
### model.fit(X_train, y_train, epochs=20, batch_size=16) # Was model.fit(X_train, y_train, nb_epoch=20, batch_size=16)


modeltype = "binary classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (14,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 37213388

Binary calssification

very Low accuracy and high loss

Commented out dataset related lines

Activation function and loss, number of output nodes, dataset labeling, learning rate

In [ ]:
# import os
# import pandas as pd
# from sklearn.cross_validation import train_test_split
from keras.models import Sequential
# from keras.layers.convolutional import Convolution2D, MaxPooling2D
from keras.layers import Dense, Activation, Dropout, Flatten
# from keras.utils import np_utils
# import AudioProcessing as ap
# import ImageTools as it
import time

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


batch_size = 128
nb_classes = 2
nb_epoch = 10  


# for i in range(20):
#     print "\n"
# if(len(os.listdir("./AudioNormalPathalogicClassification/Image")) > 0):
#     print "Audio files are already processed. Skipping..."
# else:
#     print "Generating spectrograms for the audio files..."
#     ap.audio_2_image("./AudioNormalPathalogicClassification/Audio/","./AudioNormalPathalogicClassification/Image/",".wav",".png",(28,28))

# df = pd.read_csv('./AudioNormalPathalogicClassification/Result/result.csv', header = None)
# df.columns = ["RegionName","IsNormal"]
# bool_mapping = {True : 1, False : 0}

# for col in df:
#     if(col == "RegionName"):
#         a = 3      
#     else:
#         df[col] = df[col].map(bool_mapping)
# y = df.iloc[:,1:].values
# y = np_utils.to_categorical(y, nb_classes)

# print "Loading images into memory..."
# X = it.load_images("./AudioNormalPathalogicClassification/Image/",".png")

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 0)

# X_train = X_train.reshape(X_train.shape[0], 784)
# X_test = X_test.reshape(X_test.shape[0], 784)
# X_train = X_train.astype("float32")
# X_test = X_test.astype("float32")
# X_train /= 255
# X_test /= 255

# print("X_train shape: " + str(X_train.shape))
# print(str(X_train.shape[0]) + " train samples")
# print(str(X_test.shape[0]) + " test samples")

model = Sequential()
model.add(Dense(100, input_dim = 784, activation= "relu")) # Was model.add(Dense(output_dim = 100, input_dim = 784, activation= "relu"))
model.add(Dense(200, activation = "relu")) # Was model.add(Dense(output_dim = 200, activation = "relu"))
model.add(Dense(200, activation = "relu")) # Was model.add(Dense(output_dim = 200, activation = "relu"))
model.add(Dense(nb_classes, activation = "softmax")) # Was model.add(Dense(output_dim = nb_classes, activation = "softmax"))
# + model.add(Dense(1, activation = "sigmoid"))
model.compile(loss = "categorical_crossentropy", optimizer = "adam")
# + model.compile(loss = "binary_crossentropy", optimizer = "adam")

# print model.summary()
# model.fit(X_train, y_train, batch_size = batch_size, nb_epoch = nb_epoch, show_accuracy = True, verbose = 1, validation_data = (X_test, y_test))
# score = model.evaluate(X_test, y_test, show_accuracy = True, verbose = 1)
# print("Test score: ", score[0])
# print("Test accuracy: ", score[1])


modeltype = "binary classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (784,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 37624102
Binary classification

High loss: Not reproducible

Changed Y to scaler

activation, number of output nodes

In [ ]:
import numpy as np
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten
from keras.layers import Convolution2D, MaxPooling2D
from keras.optimizers import SGD
import random
import time

A = []
B = []
for j in range(100):
    npa = np.array([[1 for j in range(100)] for i in range(100)])
    A.append(npa.reshape(npa.shape[0],npa.shape[1], 1)) # Was A.append(npa.reshape(1,npa.shape[0],npa.shape[1]))
for j in range(100):
    npa = np.array([[0 for j in range(100)] for i in range(100)])
    B.append(npa.reshape(npa.shape[0],npa.shape[1], 1)) # Was B.append(npa.reshape(1,npa.shape[0],npa.shape[1]))

trainXA = []
trainXB = []
testXA = []
testXB = []
for j in range(len(A)):
    if ((j+2) % 7) != 0:
        trainXA.append(A[j])
        trainXB.append(B[j])
    else:
        testXA.append(A[j])
        testXB.append(B[j])

X_train = np.array(trainXA + trainXB)
X_test = np.array(testXA + testXB)

Y_train = np.array([[1,0] for i in range(int(len(X_train)/2))] + [[0,1] for i in range(int(len(X_train)/2))])

# +++-- Y_train = np.array([[1] for i in range(int(len(X_train)/2))] + [[0] for i in range(int(len(X_train)/2))]) # Cahnged to scaler labels

print (X_train.shape)
print (Y_train.shape)

def jumblelists(C,D):
    outC = []
    outD = []
    for j in range(len(C)):
        newpos = int(random.random()*(len(outC)+1))
        outC = outC[:newpos]+[C[j]]+outC[newpos:]
        outD = outD[:newpos]+[D[j]]+outD[newpos:]
    return np.array(outC),np.array(outD)

X_train,Y_train = jumblelists(X_train,Y_train)


# ********************************* Added To check NORMALIZATION
# print("shape: ", X_train.shape)
# print("mean After normalization: ", np.mean(X_train, axis = 0))
# print("var After normalization: ", np.var(X_train, axis = 0))
# print("min After normalization: ", np.min(X_train, axis = 0))
# print("max After normalization: ", np.max(X_train, axis = 0))


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = Sequential()
model.add(Convolution2D(32, 3, 3, padding='valid', input_shape=(1,100,100))) # Was border_mode = 'valid'
# + model.add(Convolution2D(32, 3, 3, padding = 'valid', input_shape=(100,100, 1)))
model.add(Activation('relu'))
model.add(Convolution2D(32, 3, 3))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(128))
model.add(Activation('relu'))
model.add(Dense(2))
# + model.add(Dense(1))
model.add(Activation('softmax'))
# + model.add(Activation('sigmoid'))
sgd = SGD(0.1, decay=1e-6, momentum=0.9, nesterov=True) # Was sgd = SGD(lr=0.1, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(loss='binary_crossentropy', optimizer=sgd, metrics = ['accuracy'])
model.fit(X_train, Y_train, batch_size=32, epochs=10) # Was model.fit(X_train, Y_train, batch_size=32, nb_epoch=10)


modeltype = "binary classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (100, 100) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 38584268

Siamese neural network using CIFAR10 with 10 calsses: 60000 32x32 colour images in 10 classes

Code crashes gives error

Add a Flatten layer between the convolutional layers and the dense layer, output activation, output number of nodes  

In [ ]:
from keras.models import Sequential # Added
from keras.layers import MaxPooling2D, Convolution2D, Dense, Flatten # Added
import time

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

model = Sequential()
input_img_shape = (32, 32, 3) # Added
model.add(Convolution2D(15, 7, 7, activation='relu', padding='same', input_shape=input_img_shape)) # Was model.add(Convolution2D(15, 7, 7, activation='relu', border_mode='same', input_shape=input_img_shape))
# ------ print("C1 shape: ", model.output_shape)
model.add(MaxPooling2D((2,2), padding='same')) # Was model.add(MaxPooling2D((2,2), border_mode='same'))
# ------ print("S2 shape: ", model.output_shape)
model.add(Convolution2D(250, 5, 5, activation='relu', padding='same')) # Was model.add(Convolution2D(250, 5, 5, activation='relu', border_mode='same'))
# ------ print("C5 shape: ", model.output_shape)
# + model.add(Flatten())
model.add(Dense(50))


modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 10 # Number of classes for classification and target shape for regression
input_shape = (32, 32) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 38648195

Accuracy around 40%

Multilable classification so change loss and activation

Also random training data

In [ ]:
from sklearn.datasets import make_multilabel_classification
from sklearn.model_selection import train_test_split
# from sklearn.svm import SVC
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation
# from keras.optimizers import SGD, RMSprop
# from keras.datasets import mnist
import time

dummyX, dummyY = make_multilabel_classification(n_samples=60000, n_features=20, n_classes=3) # Increased number of samples from 4000


# ------------------------------ Added To check NORMALIZATION
# print("shape: ", X_train.shape)
# print("mean After normalization: ", np.mean(X_train, axis = 0))
# print("var After normalization: ", np.var(X_train, axis = 0))
# print("min After normalization: ", np.min(X_train, axis = 0))
# print("max After normalization: ", np.max(X_train, axis = 0))

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

model = Sequential()
model.add(Dense(20, input_dim=20))
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(20))
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(3))
model.add(Activation('softmax'))
# + model.add(Activation('sigmoid'))
model.compile(loss='mean_squared_error', optimizer='sgd', metrics=['accuracy'])
# + model.compile(loss='binary_crossentropy', optimizer='sgd', metrics=['accuracy'])
X_train, X_test, y_train, y_test = train_test_split(dummyX, dummyY, test_size=0.20, random_state=42)
model.fit(X_train, y_train,epochs=20, batch_size=30, validation_data=(X_test, y_test))


modeltype = "Multilabel Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 3 # Number of classes for classification and target shape for regression
input_shape = (20,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 39217567

multivariate regression: the dataset has 256 examples: It's binary representation of 0-256. then shuffles to learn the shuffle function.

SoF says accuracy is around 50%, output the same for all. but's actualy around 10%.

Deeplocalize is no better, even though claims 97%! It's also irrelevant because removes shuffle and learns multi label classification.

The problem is: NAS (More neurons needed: 128-65-8), Adam regulizer.

Our approach changes activations to relu and removes accuracy as metric.

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
import numpy as np
import random
import time

dim = 8
X = []
for i in range(0,2**dim):
    n = [float(x) for x in bin(i)[2:]]
    X.append([0.]*(dim-len(n))+n)
y = X[:]
random.shuffle(y)
X = np.array(X)
y = np.array(y)


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


# ---------------------------- Added To check NORMALIZATION
# print("shape: ", X.shape)
# print("mean After normalization: ", np.mean(X, axis = 0))
# print("var After normalization: ", np.var(X, axis = 0))
# print("min After normalization: ", np.min(X, axis = 0))
# print("max After normalization: ", np.max(X, axis = 0))



model = Sequential()
model.add(Dense(128, input_dim=dim, kernel_initializer='random_uniform', activation='sigmoid'))
# + model.add(Dense(128, input_dim=dim, kernel_initializer='random_uniform', activation='relu'))
model.add(Dense(64, kernel_initializer='random_uniform', activation='sigmoid'))
# + model.add(Dense(64, kernel_initializer='random_uniform', activation='relu'))
model.add(Dense(dim, kernel_initializer='random_uniform', activation='sigmoid'))
# + model.add(Dense(dim, kernel_initializer='random_uniform', activation='relu'))
model.compile(loss='mse', optimizer='adam', metrics = ['accuracy'])
model.fit(X, y, epochs=1000, batch_size=50, verbose=1)


modeltype = "Regression" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "[0,1]" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 8 # Number of classes for classification and target shape for regression
input_shape = (8,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# ---------------- Model is not trained yet
# scores = model.evaluate(X, y)
# print("%s: %.2f%%" % (model.metrics_names[1], scores[1]*100))
# output = model.predict(X)
#print("Predict: ", output)
# for i in range(0, output[:,0].size):
#     for j in range(0, output[0].size):
#         if output[i][j] > 0.5 or output[i][j] == 0.5:
#             output[i][j] = 1
#         else:
#             output[i][j] = 0
# print(y == output)

# 39525358
low accuracy, high loss

skiprow == 1

epochs from 150 to 250 (for small datsets, increase epoch to converge)

Also normalized, however it didnt help much

Reproduced dataset

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from sklearn.model_selection import train_test_split
import numpy
from sklearn.preprocessing import StandardScaler # Added
import time

seed = 7
numpy.random.seed(seed)
dataset = numpy.loadtxt("./Datasets/sorted output.csv", delimiter=",", skiprows=1) # Added skiprows == 1 to remove column names
X = dataset[:,0:3]
Y = dataset[:,3]

scaler = StandardScaler() # Added
X = scaler.fit_transform(X) # Added


# ------------------------ Added To check NORMALIZATION
# print(np.mean(X, axis = 0))
# print(np.var(X, axis = 0))
# print(np.min(X, axis = 0))
# print(np.max(X, axis = 0))

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.33, random_state=seed)
model = Sequential()
model.add(Dense(12, input_dim=3, kernel_initializer='random_uniform', activation='relu')) # Was model.add(Dense(12, input_dim=3, init='uniform', activation='relu'))
model.add(Dense(3, kernel_initializer='random_uniform', activation='relu'))
model.add(Dense(1, kernel_initializer='random_uniform', activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_train, y_train, validation_data=(X_test,y_test), epochs=250, batch_size=10) # Epochs were 150


modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (3,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 39810655

Multiclass classification

used seaborn Iris dataset: 150 samples

SoF says validation accuracy around Accuracy is 0.60, I get 30%. with 10 epochs i get 60%

epochs is not defined and the defualt value is 1.

Small dataset (150/2) so use more epochs  
Remove column names  
What about normalization?

In [ ]:
import keras ## ADDED
from keras.models import Sequential
from keras.layers import Dense, Activation
from sklearn.model_selection import train_test_split
import numpy as np
# from sklearn.datasets import load_iris
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler # Added
from sklearn.preprocessing import LabelEncoder # Added
import time

iris = sns.load_dataset("iris")
X, y = iris.iloc[:, 0:4], iris.iloc[:, 4] # Was X, y = iris.values[:, 0:4], iris.values[:, 4] and gave error: Failed to convert a NumPy array to a Tensor (Unsupported object type float).. Could also use X, y = iris.data[:, :4], iris.target

# REMOVE the first row because it contains column names
X = np.delete(X, (0), axis=0)
y = np.delete(y, (0), axis=0)


#### Added to Normalize
mms = MinMaxScaler() # Added
X = mms.fit_transform(X) # Added

train_X, test_X, train_y, test_y = train_test_split(X, y, train_size=0.5, random_state=0)

def one_hot_encode_object_array(arr):
    '''One hot encode a numpy array of objects (e.g. strings)'''
    uniques, ids = np.unique(arr, return_inverse=True)
    return keras.utils.to_categorical(ids, len(uniques)) # Was return np_utils.to_categorical(ids, len(uniques))

train_y_ohe = one_hot_encode_object_array(train_y)
test_y_ohe = one_hot_encode_object_array(test_y)



# --------------------- Added To check NORMALIZATION
# print(train_X.shape)
# print("Column means: ", np.mean(train_X, axis = 0))
# print("Column Var: ", np.var(train_X, axis = 0))
# print("Min is: ", np.min(train_X, axis = 0))
# print("Max is: ", np.max(train_X, axis = 0))


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = Sequential()
model.add(Dense(16, input_shape=(4,)))
model.add(Activation('sigmoid'))
model.add(Dense(3))
model.add(Activation('softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(train_X, train_y_ohe, verbose=1, batch_size=1, epochs = 10)

modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 3 # Number of classes for classification and target shape for regression
input_shape = (4,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# --------------- Model is not trained yet
# loss, accuracy = model.evaluate(test_X, test_y_ohe, verbose=1)
# print(model.metrics_names) # Added
# print("Accuracy is {:.2f}".format(accuracy))

# 40857445
Multiclass classification with 250 classes

Data unavailable so ommited the last two lines

Add Flatten After Max pooling

Missing Dense?

In [ ]:
from keras.layers import Activation, Convolution2D, MaxPooling2D, Flatten
from keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator # Was from keras.preprocessing.image import ImageDataGenerator
import time


if __name__ == '__main__':  

    start_time = time.perf_counter()
    %run kerassource/HomaTect.ipynb
    
    # input image dimensions
    img_rows, img_cols = 225, 225
    input_shape = (img_rows, img_cols, 1)

    model = Sequential()
    model.add(Convolution2D(64, (15, 15), input_shape=input_shape, strides=(3, 3))) # Was subsample=(3, 3) added parantheses to 15, 15
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(3, 3), strides=(2, 2)))
    # + model.add(Flatten())
    # + model.add(Dense(250))

    model.compile(loss='hinge', optimizer='adadelta', metrics=['accuracy'])

    # data = ImageDataGenerator()
    # train_data = data.flow_from_directory(directory='dataset', color_mode='grayscale', target_size=(img_rows, img_cols))
    # model.fit_generator(train_data, 100, 12)

    modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
    target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
    output_shape = 250 # Number of classes for classification and target shape for regression
    input_shape = (225, 225) # Just input shape without batch size or channels
    HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
    end_time = time.perf_counter()
    print(f"It took {end_time - start_time} seconds")

# 41372874
Binary classification with 117 data points with 57% accuracy  
Normalization makes covergence faster. after 30 epochs reaches 1 which is probably overfitting. Whithout normalization, after 200 epochs is still 63%    

For small datsets we need higher epochs. so for 50 epochs it reches 100% Accuracy

In [ ]:
import numpy
import pandas
import matplotlib.pyplot as plt
from keras.models import Sequential
from keras.layers import Dense
from sklearn.preprocessing import StandardScaler # Added
import time

seed = 7
numpy.random.seed(seed)

dataframe = pandas.read_csv('./Datasets/SIN.csv', usecols=[0,1,2], engine='python')
dataset = dataframe.values
X = dataset[:,0:2].astype(float)
Y = dataset[:,2].astype(int)

train_size = int(len(X) * 0.80)
test_size = len(X) - train_size
Xtrain, Xtest = X[0:train_size,:], X[train_size:len(X),:]
Ytrain, Ytest = Y[0:train_size], Y[train_size:len(Y)]

print(len(Xtrain), len(Xtest))



scaler = StandardScaler() # Added
Xtrain = scaler.fit_transform(Xtrain) # Added


# ------------------- Added To check NORMALIZATION
# print("shape: ", Xtrain.shape)
# print("mean After normalization: ", numpy.mean(Xtrain, axis = 0))
# print("var After normalization: ", numpy.var(Xtrain, axis = 0))
# print("min After normalization: ", numpy.min(Xtrain, axis = 0))
# print("max After normalization: ", numpy.max(Xtrain, axis = 0))


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = Sequential()
model.add(Dense(2,  kernel_initializer='random_uniform', input_dim=2, activation='relu')) # Was model.add(Dense(2, input_dim=2, init='uniform', activation='relu'))
model.add(Dense(2,  kernel_initializer='random_uniform', activation='relu')) # Was model.add(Dense(2, init='uniform', activation='relu')) 
model.add(Dense(1,  kernel_initializer='random_uniform', activation='sigmoid')) #Was model.add(Dense(1, init='uniform', activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(Xtrain, Ytrain, epochs=50, batch_size=2, verbose=2) # epochs = 20


modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (2,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")


# ------- Model is not trained yet. Estimate model performance
# trainScore = model.evaluate(Xtrain, Ytrain, verbose=2)
# print('Train Score: %.2f' % trainScore[1])
# testScore = model.evaluate(Xtest, Ytest, verbose=2)
# print('Test Score: %.2f' % testScore[1])

# 41600519

Binary Classification with Higgs Boson challenge on kaggle.

Accuracy bellow 70%- model always predicts the majority class

Problem is unbalanced dataset, optimizer and NAS.

HomaTect cant detect

In [ ]:
from keras.layers import Dense, Activation, Dropout
from keras.models import Model
from keras.models import Sequential
from keras.optimizers import SGD
import time

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

model = Sequential()
model.add(Dense(600, input_shape=(30,),activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(400, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(100, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))
sgd = SGD(0.01, decay=1e-6)
model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])
#model.fit(train,labels,nb_epoch=1,batch_size=1)

modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (30,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 42913869
Regression problem with 5 integers and 3 outputs

I intent to do some Q learning using neural networks. Hence, the input of my network is a state s (described by an array of 5 integers) and the results is an array of the values Q(s,a) where a can take three values (0,1, or 2)

Crashes with error message: "Expected to see 1 arrays but instead got the following list of 2 arrays: ...",
I get: Invalid input shape for input Tensor("sequential_49_1/Cast:0", shape=(1,), dtype=float32). Expected shape (None, 2), but input has incompatible shape (1,)

Cant reproduce results without a dataset

Number of output nodes should be 3

input_dim should be 5

For loop must be ommited: Should state you are training on one data sample!
fit arguments for x and y is incorrect

In [ ]:
import numpy as np # Added
from keras.models import Sequential # Added
from keras.layers import Dense # Added
import time

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

model = Sequential()
model.add(Dense(10, input_dim=1, activation='relu'))
# + model.add(Dense(10, input_dim=5, activation='relu'))
model.add(Dense(10, activation='relu'))
model.add(Dense(10, activation='relu'))
model.add(Dense(1))
# + model.add(Dense(3))
model.compile(loss='mse', optimizer='adam', metrics = ['accuracy']) # Accuracy not a good metric
data = np.array([x for x in range(0,1000)]) # random data with incorrect dimentioality
for i in range(0, 1000):
   model.fit(np.array([data[i]]), np.array([data[i]]), epochs=1, batch_size=1, verbose=0) # Was model.fit([np.array([data[i]]), np.array([data[i]]), nb_epoch=1, batch_size=1, verbose=0)

modeltype = "regression" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "[-n,n]" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 3 # Number of classes for classification and target shape for regression
input_shape = (5,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 43464835

Training dataset shape: (300, 5, 720)

Crashed with error message: ValueError: Error when checking model input: expected conv2d_1_input to have 4 dimensions, but got array with shape (300, 5, 720)

This is one of the most classic errors in Convolutions in Keras. Its origin lies in the fact that when using channels_last input dimension you need to make your input to have dimension (height, width, channels) even in case when you have only one channel. So basically reshape using x_train = x_train.reshape((x_train.shape[0], 5, 720, 1)

fit commented out cause we dont have training data  
But shape error is actually in fit because we have not reshaped the dataset. just correctly defined input_shape

In [ ]:
from keras.models import Sequential # Added
from keras.layers import Dense, Conv2D, MaxPooling2D, Dropout, Flatten # Added
import time

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

number_of_classes = 3 # Not clear
x_train_shape = (300, 5, 720)
rows, cols=x_train_shape[1:] # Was x_train.shape[1:] but changed cause we dont have dataset
print(rows, cols)
in_shape=(rows,cols,1)
print(in_shape)
cnn = Sequential()
cnn.add(Conv2D(64, (5, 50), padding="same", activation="relu",data_format="channels_last", input_shape=in_shape))
cnn.add(MaxPooling2D(pool_size=(2,2),data_format="channels_last"))
cnn.add(Flatten())
cnn.add(Dropout(0.5))
cnn.add(Dense(number_of_classes, activation="softmax"))
cnn.compile(loss="categorical_crossentropy", optimizer="adam", metrics= ['accuracy'])

modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 3 # Number of classes for classification and target shape for regression
input_shape = (5,720) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# x_train = x_train.reshape((x_train.shape[0], 5, 720, 1) # Actually this should be added
# cnn.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_data=(x_test, y_test), shuffle=True)

# 44066044
a binary classification algorithm (output is 0 or 1) on a dataset that contains normal and malicious network packets

Dataset from Deeplocalize

Accuracy stuck at 50% - reproduce with simulated dataset gives 61.9%

changed activation and scaled

actually scaling increased accuracy to 100%

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
# from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler # Added
import numpy
import os
import time

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
seed = 4
numpy.random.seed(seed)

dataset = numpy.loadtxt("./Datasets/NetworkPackets.csv", delimiter=",")
X = dataset[:, 0:11].astype(float)
Y = dataset[:, 11]

scaler = StandardScaler() # Added
X = scaler.fit_transform(X) # Added




#------------- Added To check NORMALIZATION
# print("shape: ", X.shape)
# print("mean After normalization: ", np.mean(X, axis = 0))
# print("var After normalization: ", np.var(X, axis = 0))
# print("min After normalization: ", np.min(X, axis = 0))
# print("max After normalization: ", np.max(X, axis = 0))


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = Sequential()
model.add(Dense(12, input_dim=11, kernel_initializer='normal', activation='relu'))
model.add(Dense(12, kernel_initializer='normal', activation='relu'))
model.add(Dense(1, kernel_initializer='normal', activation='relu'))
# + model.add(Dense(1, kernel_initializer='normal', activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='Adam', metrics=['accuracy'])
model.fit(X, Y, epochs=100, batch_size=5)

modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (11,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# ------------ Model is not trained yet
# scores = model.evaluate(X, Y)
# print("\n%s: %.2f%%" % (model.metrics_names[1], scores[1]*100))

# 44164749
Multilable classification with 11 classes

no apparent bug reported

Deep localise uses mnist dataset which is irrelevant to multilabel classification  

activation and loss  
Also data preprocessing is required

In [ ]:
import keras # Added
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation
from keras.optimizers import SGD
from keras.datasets import mnist # Added
from keras.initializers import RandomNormal # Added
import time

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = Sequential()
model.add(Dense(5000, activation='relu', input_dim=784)) # WAS model.add(Dense(5000, activation='relu', input_dim=x_train.shape[1]))
model.add(Dropout(0.1))
model.add(Dense(600, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(11, activation='softmax')) # WAS Was model.add(Dense(y_train.shape[1], activation='softmax'))
# + model.add(Dense(11, activation='sigmoid'))

sgd = SGD(0.01, decay=1e-6, momentum=0.9, nesterov=True) # Was sgd = SGD(lr=0.01, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy',])
# + model.compile(loss='binary_crossentropy', optimizer=sgd, metrics=['accuracy',])

#model.fit(x_train, y_train,epochs=5,batch_size=2000)
#score = model.evaluate(x_test, y_test, batch_size=2000)
#preds = model.predict(x_test)
#preds[preds>=0.5] = 1
#preds[preds<0.5] = 0

modeltype = "Multilabel Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 11 # Number of classes for classification and target shape for regression
input_shape = (784,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 44322611

Binary classification: Orange, apple dataset, images of 224x224 pixels

Accuracy stuck at 50% and predicts the same result every time

User's Answer: Because of the huge number of neurons used for classifying just 2 classes, the structure was overfitting, causing the accuracy to stick at 50%. The second problem was with the sgd optimizer. Used RMSProp instead of SGD

The author doesnt mention loss as an error but it is 

Dont have access to dataset so deleted dataset related lines

In [ ]:
# import numpy as np
# from keras.datasets import cifar10
# from datasetFetch import DataFetch
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
# from keras.constraints import maxnorm
from keras.optimizers import SGD
# from keras.utils import np_utils
# from keras import backend as K
# K.set_image_dim_ordering('th')
# import simplejson
# from matplotlib import pyplot
# from scipy.misc import toimage
import time

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

num_classes = 2
#Removed data preproccessing steps cause dont have access to dataset

model = Sequential()
model.add(Conv2D(224, (11, 11), input_shape=(224, 224, 3), activation='relu', padding='same'))
model.add(Dropout(0.2))
model.add(Conv2D(55, (5, 5), activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2, 2), data_format="channels_last"))
model.add(Conv2D(13, (3, 3), activation='relu', padding='same'))
model.add(Dropout(0.5))
model.add(Conv2D(13, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2, 2), data_format="channels_last"))
model.add(Flatten())
model.add(Dropout(0.2))
model.add(Dense(512, activation='relu')) # model.add(Dense(512, activation='relu', kernel_constraint=maxnorm(3)))
model.add(Dropout(0.2))
model.add(Dense(2, activation='softmax'))
# + model.add(Dense(1, activation='sigmoid'))

epochs = 100
lrate = 0.01
decay = lrate/epochs
sgd = SGD(lrate, momentum=0.9, decay=decay, nesterov=False)
model.compile(loss='binary_crossentropy', optimizer=sgd, metrics=['accuracy'])
# model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, batch_size=10)
# scores = model.evaluate(X_test, y_test, verbose=0)
# print("Accuracy: %.2f%%" % (scores[1]*100))


modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (224,224) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 44998910

Regression: generated some data from a 4th degree polynomial

Poor accuracy, predictions seem to be linear

Dataset not available: 9900 samples

more epochs, hidden layer nonlinear activation  

The author said sigmoid worked better

Probaby data standardization too

In [ ]:
from keras.models import Sequential # Added
from keras.layers import Dense, Activation # Added
import time

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

model = Sequential()
model.add(Dense(units=200, input_dim=1))
model.add(Activation('relu'))
model.add(Dense(units=45))
model.add(Activation('linear'))
model.add(Dense(units=1))
model.compile(loss='mean_squared_error', optimizer='sgd')
#model.fit(x_train, y_train, epochs=20, batch_size=50)
#loss_and_metrics = model.evaluate(x_test, y_test, batch_size=100) # Datset not available
#prediction = model.predict(x_test, batch_size=1) # Dataset not available


modeltype = "regression" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "[-n,n]" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 1 # Number of classes for classification and target shape for regression
input_shape = (1,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 45120429

Dataset not available: 32byte array of numbers, Boolean output

Crashes with error: ValueError: Error when checking target: expected activation_5 to have 3 dimensions, but got array with shape (1000, 1)

Flatten before Dense

Dense layer need 1 output unit

for the last convolution Only use one filter so that the output will be a sequence of 28 values, not a matrix

In [ ]:
import keras as k
# import numpy as np
import time

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

# inputData = np.array(arr1)
# inputData = np.expand_dims(inputData, axis = 2)
# labelData = np.array(arr2)

# print inputData.shape
# print labelData.shape
model = k.models.Sequential()
model.add(k.layers.Convolution1D(32,2, input_shape = (32, 1))) # Was model.add(k.layers.convolutional.Convolution1D(32,2, input_shape = (32, 1)))
model.add(k.layers.Activation('relu'))
model.add(k.layers.Convolution1D(32,2)) # Was model.add(k.layers.convolutional.Convolution1D(32,2))
model.add(k.layers.Activation('relu'))
model.add(k.layers.Convolution1D(32,2)) # Was model.add(k.layers.convolutional.Convolution1D(32,2))
model.add(k.layers.Activation('relu'))
model.add(k.layers.Convolution1D(32,2)) # Was model.add(k.layers.convolutional.Convolution1D(32,2))
# ---+ model.add(k.layers.Convolution1D(1,2))
model.add(k.layers.Activation('relu'))
# + model.add(k.layers.Flatten())
model.add(k.layers.Dense(32)) # Was model.add(k.layers.core.Dense(32))
# + model.add(k.layers.Dense(1))
model.add(k.layers.Activation('sigmoid'))
model.compile(loss = 'binary_crossentropy', optimizer = 'rmsprop', metrics=['accuracy'])
## model.fit( inputData,labelData ) # Dataset not available


modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (32,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")


# ---- model.summary()

# 45337371

Seems to be binary classification because of one output unit

Softmax is the problem

In [ ]:
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense
# from keras import backend as K
import numpy as np #Added
import time

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

num_features = 5
model = Sequential()
model.add(Dense(60, input_shape=(num_features,), activation='relu'))
model.add(Dense(50, activation='relu'))
model.add(Dense(1, activation='softmax'))
# + model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy')
# model.predict(np.random.rand(1, num_features))

# x = tf.random.uniform(shape=(1, num_features)) # Was x = tf.random_uniform(shape=(1, num_features))
# model_grad = tf.GradientTape(model(x), x) # Was model_grad = tf.gradients(model(x), x)[0]

modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (5,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 45378493

Binary classification using CNN

Dataset not available: 6000 samples with (10, 4) dimensions

Commented out dataset related linnes

Always predicts one

Change softmax to sigmoid

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Convolution1D, BatchNormalization, LeakyReLU, Dropout, Flatten, Activation
from keras.callbacks import ReduceLROnPlateau # Added
import time

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

input_shape = (10, 4) # 4 will be considered channels
model = Sequential()
model.add(Convolution1D(input_shape = input_shape, filters=16, kernel_size=4, padding='same')) # Was model.add(Convolution1D(input_shape = (10, 4), nb_filter=16, filter_length=4, border_mode='same'))
model.add(BatchNormalization())
model.add(LeakyReLU())
model.add(Dropout(0.2))
model.add(Convolution1D(filters=8, kernel_size=4, padding='same')) # model.add(Convolution1D(nb_filter=8, filter_length=4, border_mode='same'))
model.add(BatchNormalization())
model.add(LeakyReLU())
model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(64))
model.add(BatchNormalization())
model.add(LeakyReLU())
model.add(Dense(1))
model.add(Activation('softmax'))
# + model.add(Activation('sigmoid'))

reduce_lr = ReduceLROnPlateau(monitor='val_acc', factor=0.9, patience=30, min_lr=0.000001, verbose=0)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# history = model.fit(X_train, y_train, epochs = 100, batch_size = 128, verbose=0, validation_data=(X_test, y_test), callbacks=[reduce_lr], shuffle=True)
# y_pred = model.predict(X_test)

modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (10,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 45442843
Binary classification
Dataset: values ranging from 15000 to 25000. approx. 20000 will end up in category 1 and the rest in category 2

Code and Dataset from DeepLocalize

All values end up in category 2

last dense layer units = 1, Ativation = Sigmoid, loss = binary_crossentropy

Also standardized data  
Better to use oroginal code not Deeplocalize's

In [ ]:
from keras.models import Sequential # Added
from keras.layers import Dense, Activation # Added
import numpy # Added
import pandas # Added
from sklearn import preprocessing # Added
import time

# Data Pre-proccessing added
seed = 7
numpy.random.seed(seed)
dataframe = pandas.read_csv('./Datasets/2000.csv', delimiter=",")
dataset = dataframe.values
X = dataset[:,0:1].astype(float)
Y = dataset[:,1].astype(bool)
X = preprocessing.scale(X)
train_size = int(len(X) * 0.80)
test_size = len(X) - train_size
Xtrain, Xtest = X[0:train_size,:], X[train_size:len(X),:]
Ytrain, Ytest = Y[0:train_size], Y[train_size:len(Y)]



#-------------------------- Added To check NORMALIZATION
# print("shape: ", Xtrain.shape)
# print("mean After normalization: ", numpy.mean(Xtrain, axis = 0))
# print("var After normalization: ", numpy.var(Xtrain, axis = 0))
# print("min After normalization: ", numpy.min(Xtrain, axis = 0))
# print("max After normalization: ", numpy.max(Xtrain, axis = 0))

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = Sequential()
model.add(Dense(1 , input_dim =1))
model.add(Activation('sigmoid'))
# + model.add(Activation('relu'))
model.add(Dense(2 , kernel_initializer='random_uniform' , activation = 'softmax')) # Was model.add(Dense(2 , init='normal' , activation = 'softmax'))
# + model.add(Dense(1 , kernel_initializer='random_uniform' , activation = 'sigmoid'))
model.compile(loss='mean_absolute_error', optimizer='rmsprop', metrics=['accuracy'])
# + model.compile(loss='binary_crossentropy', optimizer='rmsprop', metrics=['accuracy'])
model.fit(Xtrain, Ytrain, validation_data=(Xtest, Ytest),epochs=10,batch_size=200,verbose=2)
# + model.fit(Xtrain, Ytrain, validation_data=(Xtest, Ytest),epochs=100,batch_size=200,verbose=2)


modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (1,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")



# -------------- Model is not trained yet
# model.evaluate(Xtest,Ytest)

# 45711636

Crashes with error: ValueError: ('The specified size contains a dimension with value <= 0', (-192, 1024))

My error: Computed output size would be negative. Received `inputs shape=(None, 1, 20, 56)`, `kernel shape=(3, 3, 56, 32)`, `dilation_rate=[1 1].  
you can either set the corresponding parameter in the keras.json config file to th or channels_first so it matches your file, or you change the input_shape of your data to (20, 56, 1)

change the input_shape of your data to (20, 56, 1) because of data_format.

commednted out dataset related lines

In [ ]:
from keras.models import Sequential # Added
from keras.layers import Dense, Conv2D, Dropout, MaxPooling2D, Flatten
import time

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

num_classes = 10 # Added randomly
model = Sequential()
model.add(Conv2D(32, (3, 3), padding='valid', activation='relu',input_shape=(1, 20, 56)))
# + model.add(Conv2D(32, (3, 3), padding='valid', activation='relu',input_shape=(20, 56, 1)))
model.add(Dropout(0.25))
model.add(Conv2D(32, (3, 3), padding='valid', activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Conv2D(32, (3, 3), padding='valid', activation='relu')) # Was model.add(Conv2D(32, (3, 3), border_mode='valid', activation='relu'))
model.add(Conv2D(32, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(1024, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# model_info = model.fit(X_3, y_3, batch_size=100, epochs=20, verbose=2, validation_data=(X_test, y_test))

modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 10 # Number of classes for classification and target shape for regression
input_shape = (20,56) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 46177505

Binary classification  
Crashes with error: Input dimension mis-match

My error: Computed output size would be negative. Received `inputs shape=(None, 0, 99, 32)`, `kernel shape=(3, 3, 32, 32)`, `dilation_rate=[1 1]`.

Commented out unneccessary lines

nb_channel: It should be input_shape=(img_rows, img_cols, nb_channel) because we are not useing theano backend

The Author said the problem was the shape of the custom_gabor return and it is.  

number of output nodes should be 1, output activation should be sigmoid

In [ ]:
import random
# from cv2.cv2 import CV_64F
# import keras.backend as K
import numpy as np
# from sklearn.model_selection import train_test_split
# from sklearn.utils import shuffle
# from keras.utils import np_utils
from keras.layers import Dense, Dropout, Activation, Flatten, MaxPooling2D, Conv2D
from keras.models import Sequential
# import cv2
# from theano import shared
import time

images_size = 100
img_rows = 200
img_cols = 200
nb_channel = 3


# def custom_gabor(shape, dtype=None):
#    total_ker = []
#    for i in xrange(shape[3]):
#        kernels = []
#        for j in xrange(shape[2]):
#            kernels.append(
#            cv2.getGaborKernel(ksize=(shape[0], shape[1]), sigma=1, 
#            theta=1, lambd=0.5, gamma=0.3, psi=(3.14) * 0.5, 
#            ktype=CV_64F))
#        total_ker.append(kernels)
#    np_tot = shared(np.array(total_ker))
#    return K.variable(np_tot, dtype=dtype)


def build_model():
    model = Sequential()
    model.add(Conv2D(32, (3, 3), input_shape=(nb_channel,img_rows, img_cols))) # we are not using theano backend to use channel first so Was kernel_initializer=custom_gabor & input_shape=(nb_channel, img_rows, img_cols)
    # + model.add(Conv2D(32, (3, 3), input_shape=(img_rows, img_cols, nb_channel)))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(32, (3, 3))) # Was kernel_initializer=custom_gabor
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(32, (3, 3))) # Was kernel_initializer=custom_gabor
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))
    model.add(Flatten())
    model.add(Dense(64))
    model.add(Activation('relu'))
    model.add(Dropout(0.5))
    model.add(Dense(2))
    # + model.add(Dense(1))
    model.add(Activation('softmax'))
    # + model.add(Activation('sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model


# def make_dummy_data_set():
#    img_matrix = []
#    for i in xrange(images_size):
#        img_matrix.append([random.random() for _ in xrange(img_rows*img_cols*nb_channel)])

#    img_matrix = np.array(img_matrix)
#    label = np.array([random.randint(0, 1) for _ in xrange(images_size)])

#    data, label = shuffle(img_matrix, label, random_state=7)  # random_state=2
#    X_train, X_test, y_train, y_test = train_test_split(data, label, test_size=0.2, random_state=7)

#    X_train = X_train.reshape(X_train.shape[0], nb_channel, img_rows, img_cols)
#    X_test = X_test.reshape(X_test.shape[0], nb_channel, img_rows, img_cols)

#    X_train = X_train.astype('float32')
#    X_test = X_test.astype('float32')

#    y_train = np_utils.to_categorical(y_train, 2)
#    y_test = np_utils.to_categorical(y_test, 2)

#    return X_train, X_test, y_train, y_test


# def train_model(model, X_train, X_test, y_train, y_test):
#    model.fit(X_train, y_train, batch_size=32, epochs=5, verbose=1, validation_data=(X_test, y_test))


if __name__ == "__main__":

    start_time = time.perf_counter()
    %run kerassource/HomaTect.ipynb
    
    model = build_model()
#    X_train, X_test, y_train, y_test = make_dummy_data_set()
#    train_model(model, X_train, X_test, y_train, y_test)

    modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
    target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
    output_shape = 2 # Number of classes for classification and target shape for regression
    input_shape = (200,200) # Just input shape without batch size or channels
    HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
    end_time = time.perf_counter()
    print(f"It took {end_time - start_time} seconds")

# 46995209
Regression: one-dimensional sine

plots dont match, loss around 37% becomes 0.07% after fix

Accuracy not a good metric for regression

Standardized data

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
import numpy
import sklearn.datasets
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
from sklearn import preprocessing # Added
import time

numpy.random.seed(7)
T = 1000
X = numpy.array(range(T))
Y = numpy.sin(3.5 * numpy.pi * X / T)
X = preprocessing.scale(X) # Added
# Before preprocessing: [0,1000] and after: [-1.7, 1.7] 

plt.scatter(X, Y, s = 10)
plt.show()

input_dim = 1


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = Sequential()
model.add(Dense(10, input_dim = input_dim, activation='tanh'))
model.add(Dense(90, activation='tanh'))
model.add(Dense(10, activation='tanh'))
model.add(Dense(10, activation='tanh'))
model.add(Dense(10, activation='tanh'))
model.add(Dense(10, activation='tanh'))
model.add(Dense(10, activation='tanh'))
model.add(Dense(10, activation='tanh'))
model.add(Dense(10, activation='tanh'))
model.add(Dense(10, activation='tanh'))
model.add(Dense(10, activation='tanh'))
model.add(Dense(1, activation='tanh'))

model.compile(loss='mean_squared_error', optimizer='adam', metrics=['accuracy']) # Remove accuracy as metric
model.fit(X, Y, epochs=50, batch_size=10)


modeltype = "Regression" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "[-1,1]" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 1 # Number of classes for classification and target shape for regression
input_shape = (1,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")


# -------------- Model is not trained yet
#scores = model.evaluate(X, Y)
#print("-" * 100)
#print("\n%s: %.2f%%" % (model.metrics_names[0], scores[0]*100))

#pred =  model.predict(X)
#x_plot = X
#plt.scatter(x_plot, pred, s = 1, c =  'r')
#plt.scatter(x_plot, Y, s = 1, c = 'b')
#plt.show()

# 47352366

Mnist multiclass classification

Too slow

Commented out lines related to dataset

Kernel size from 5*5 to (5, 5), 
activation to softmax

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Reshape, Conv2D, MaxPooling2D, Dropout, Flatten, Dense
import datetime
import time

# from tensorflow.examples.tutorials.mnist import input_data # Removed
# mnist = input_data.read_data_sets('MNIST_data', one_hot=True) # Removed

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

model = Sequential()
bias_initializer = keras.initializers.Constant(value = 0.1)
neurons_nb_layer_1 = 32
neurons_nb_layer_2 = 64
neurons_nb_layer_3 = 1024
model.add(Reshape((28, 28, 1), input_shape=(784,)))
model.add(Conv2D(filters = neurons_nb_layer_1, kernel_size = 5*5, padding = 'same', activation = "relu", bias_initializer = bias_initializer))
# + model.add(Conv2D(filters = neurons_nb_layer_1, kernel_size = (5, 5), padding = 'same', activation = "relu", bias_initializer = bias_initializer))
model.add(MaxPooling2D(padding='same'))
model.add(Conv2D(filters = neurons_nb_layer_2, kernel_size = 5*5, padding = 'same', activation = "relu", bias_initializer = bias_initializer))
# + model.add(Conv2D(filters = neurons_nb_layer_2, kernel_size = (5, 5), padding = 'same', activation = "relu", bias_initializer = bias_initializer))
model.add(MaxPooling2D(padding='same'))
model.add(Reshape((1,7*7*neurons_nb_layer_2)))
model.add(Dense(units = neurons_nb_layer_3, activation = "relu", bias_initializer = bias_initializer))
model.add(Dropout(rate = 0.5))
model.add(Flatten())
model.add(Dense(units = 10, activation = "relu"))
# + model.add(Dense(units = 10, activation = "softmax"))
# model.summary()
model.compile(loss = keras.losses.categorical_crossentropy, optimizer = 'adam', metrics=['accuracy'])

#start2 = datetime.datetime.now()
#for i in range(600):
#    batch = mnist.train.next_batch(50)
#    if i % 100 == 0:
#        train_accuracy = model.evaluate(batch[0], batch[1])
#        print("step", i, ":", train_accuracy)
#    model.train_on_batch(batch[0], batch[1])
#end2 = datetime.datetime.now()
#time2 = (end2 - start2).seconds
#print(time2//60, "min", time2%60,"s")


modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 10 # Number of classes for classification and target shape for regression
input_shape = (28,28) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 47724077


Binary classification with Adult dataset: if income exceeds 50k

But implemented as a categorical classification with one hot encoding which seesms to be fine.

Crash with error: "ValueError: could not broadcast input array from shape (14,1) into shape (14)"

My error: could not broadcast input array from shape (42,32561) into shape (1,32561) because of data preproccessing. Fixed this error.

The author said updating to a newer version keras solves the issue

Deeplocalize's code doesnt even run. The suggeted fix (change optimizer to sgd) doesnt make any difference in accuracy.  
There are three numerical columns which are not normalized  

In [ ]:
import pandas as pd # Added
from keras.models import Sequential # Added
from keras.layers import Dense # Added
from sklearn.preprocessing import LabelEncoder # Added
import keras
import numpy as np # Added
import time

df = pd.read_csv("./Datasets/Adult/adult.csv",header=None)
XX = df.iloc[:,0:14]
YY = df.iloc[:,14]
##########columns = XX.columns # Added
XX = np.array(XX) # Added

# encoder = LabelEncoder() # move inside
for i in [1,3,5,6,7,8,9,13]:
    #column = XX[i]
    encoder = LabelEncoder() # Added
    encoder.fit(list(XX[:,i])) # Was encoder.fit(column)
    XX[:,i] = encoder.transform(XX[:,i]) # Was encoded_C = encoder.transform(column)
    #X[i] = np_utils.to_categorical(encoded_C)

XX = XX.astype(np.float32) # Added

encoder.fit(YY)
en_Y = encoder.transform(YY)
YY = keras.utils.to_categorical(en_Y) # Was np_utils.to_categorical(en_Y)

print(YY.shape)

#----------------- Added To check NORMALIZATION
# print("shape: ", XX.shape)
# print("mean After normalization: ", np.mean(XX, axis = 0))
# print("var After normalization: ", np.var(XX, axis = 0))
# print("min After normalization: ", np.min(XX, axis = 0))
# print("max After normalization: ", np.max(XX, axis = 0))


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = Sequential()
model.add(Dense(21, input_dim=14, activation="relu"))
model.add(Dense(2, activation="softmax"))
model.compile(loss="categorical_crossentropy",optimizer="adam",metrics= ["accuracy"]) # DeepLocalize change optimizer to sgd!
model.fit(XX,YY, epochs=50, batch_size=100)


modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (14,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")


# ------------ Model is not trained
#score = model.evaluate(XX,YY)
#print(model.metrics_names) # Added
#print("Accuracy: {}%".format(score[0])) # it's loss not accuracy

# 48221692

Regression problem

Datset were f(x) = x^2 + 3

outputs ranging from 0 to 1

Change outputs ranging from 0 to 1 output activation to linear

After standardization, plots become weired!!! so changed prediction line

Hidden layer activation shouldnt be sigmoid, if it'srelu plots match better, tanh is the best. with sigmoid they dont match

In [ ]:
import numpy # Added
import pandas as pd # Added
from keras.models import Sequential # Added
from keras.layers import Dense # Added
from sklearn.model_selection import train_test_split # Added
import matplotlib # Added to plot
matplotlib.use("TkAgg") # Added to plot
import matplotlib.pyplot as plt # Added to plot
from sklearn.preprocessing import StandardScaler # Added
import time

dataset = pd.read_csv("./Datasets/Linear_Data.csv", header=None).values
X_train, X_test, Y_train, Y_test = train_test_split(dataset[:,0:1], dataset[:,1], test_size=0.25,)
print(Y_test)

# Standardization Added but removed because plot became weired!
sc_x = StandardScaler()
sc_y = StandardScaler()
X_train = sc_x.fit_transform(X_train)
X_test = sc_x.transform(X_test)
Y_train = Y_train.reshape(-1,1)
Y_test = Y_test.reshape(-1,1)
Y_train = sc_y.fit_transform(Y_train)
Y_test = sc_y.transform(Y_test)


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


neural_network = Sequential()
neural_network.add(Dense(5, input_dim=1, activation='sigmoid'))
# + neural_network.add(Dense(5, input_dim=1, activation='tanh'))
neural_network.add(Dense(1, activation='sigmoid'))
# + neural_network.add(Dense(1, activation='linear'))
neural_network.compile(loss='mean_squared_error', optimizer='sgd', metrics=['accuracy'])
neural_network_fitted = neural_network.fit(X_train, Y_train, epochs=1000, verbose=0, batch_size=X_train.shape[0], initial_epoch=0)


modeltype = "Regression" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "[0, n]" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 1 # Number of classes for classification and target shape for regression
input_shape = (1,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")



# Added to plot
pred = neural_network.predict(X_train) # Was pred = sc_y.inverse_transform(neural_network.predict(sc_x.transform(X_train)))

x_plot = X_train
plt.scatter(x_plot, pred, s = 1, c =  'r')
plt.scatter(x_plot, Y_train, s = 1, c = 'b')
plt.show()

# 48251943

Dataset consists of negative and positive but NN doest predict negative values  

DeepLocalizes dataset is irrelevant. it made a dataset out of target values presented in SoF post.  

Tanh instead of relu

Also Data preproccessing error that we cant detect

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
import numpy
import time

numpy.random.seed(7)

# dataset = numpy.loadtxt("./Datasets/Book1.csv", delimiter=";")
dataset = numpy.genfromtxt("./Datasets/Book1.csv", delimiter=',') # Because datset contains missing values, Also change Delimiter to ","

X = dataset[:,2:4]
Y = dataset[:,1]
print ("Variables: \n", X)
print ("Target_outputs: \n", Y)


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = Sequential()
model.add(Dense(4, input_dim=2, activation='relu'))
model.add(Dense(1, activation='relu'))
# + model.add(Dense(1, activation='tanh'))
# model.summary()

model.compile(loss='mean_squared_error', optimizer='sgd', metrics=['MSE'])
model.fit(X, Y, epochs=500, batch_size=10)


modeltype = "Regression" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "[-1, 1]" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 1 # Number of classes for classification and target shape for regression
input_shape = (1,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")



# ---------------- Model is not trained yet
#F = model.predict(X)
#print ("Predicted values: \n", F)

# 48385830


Multiclass classification with MNIST dataset

loss: nan, accuracy: 0.098

No activation defined

loss to categorical_crossentropy

In [ ]:
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers import SGD
from keras.initializers import RandomNormal
import keras
import time

(x_train, y_train), (x_test, y_test) = mnist.load_data()

img_rows, img_cols = 28, 28
x_train = x_train.reshape(x_train.shape[0], img_rows * img_cols)
x_test = x_test.reshape(x_test.shape[0], img_rows * img_cols)
input_shape = (img_rows * img_cols,)

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255
print('x_train shape:', x_train.shape)
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

num_classes = 10
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)
print('y_train shape:', y_train.shape)


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = Sequential()
layer1 = Dense(30, input_shape=input_shape, kernel_initializer=RandomNormal(stddev=1), bias_initializer=RandomNormal(stddev=1))
# + layer1 = Dense(30, input_shape=input_shape, kernel_initializer=RandomNormal(stddev=1), bias_initializer=RandomNormal(stddev=1), activation='relu')
model.add(layer1)
layer2 = Dense(10, kernel_initializer=RandomNormal(stddev=1), bias_initializer=RandomNormal(stddev=1))
# + layer2 = Dense(10, kernel_initializer=RandomNormal(stddev=1), bias_initializer=RandomNormal(stddev=1), activation='softmax')
model.add(layer2)

# model.summary()
model.compile(optimizer=SGD(3.0), loss='mean_squared_error', metrics=['accuracy'])
# + model.compile(optimizer=SGD(3.0), loss='categorical_crossentropy', metrics=['accuracy'])

model.fit(x_train, y_train, batch_size=10, epochs=30, verbose=2)



modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 10 # Number of classes for classification and target shape for regression
input_shape = (28,28) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")


# ------------ Model is not trained yet
#result = model.evaluate(x_test, y_test, verbose=1)
#print('Test: ', result)

# 48594888

Multiclass classification: Overfitting (High train accuracy, low test accuracy)

SoF suggestions:

he_normal initializer is better than glorot_uniform

It is strange to decrease the number of filters as you go deeper in the network. You should do right the opposite. I changed 256 -> 64 and 128 -> 256 and the accuracy improved

I decreased the dropout slightly 0.5 -> 0.4

Kernel size 3x3 is more common than 2x2

normalize data


HomaTect can detect two important ones: Main issue is normalization. without normalization, it's 55% increases to 66%. with filters, furthur increases to 69%


In [ ]:
from keras.datasets import cifar10
from keras.models import Sequential
from keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout
import keras
import time

(x_train, y_train), (x_test, y_test) = cifar10.load_data()
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)
x_train = x_train.astype('float32') # Added
x_test = x_test.astype('float32') # Added
x_train /= 255 # Added
x_test /= 255 # Added

start_time = time.perf_counter()
#%run kerassource/HomaTect.ipynb


model = Sequential()
model.add(Conv2D(filters=256, kernel_size=(3, 3), activation='relu', data_format='channels_last', input_shape=(32, 32, 3)))
# + model.add(Conv2D(filters=64, kernel_size=(3, 3), activation='relu', input_shape=(32, 32, 3)))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(filters=128, kernel_size=(2, 2), activation='relu'))
# + model.add(Conv2D(filters=256, kernel_size=(2, 2), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Flatten())
model.add(Dense(1024, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(10, activation='softmax'))
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=['accuracy']) # Was model.compile(optimizer=adam(), loss=categorical_crossentropy, metrics=['accuracy'])
model.fit(x_train, y_train, batch_size=1000, epochs=5, verbose=1, validation_data=(x_test, y_test)) # WAS batch_size=1000

modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 10 # Number of classes for classification and target shape for regression
input_shape = (32,32) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")


# -------------- Model is not trained yet
#loss, accuracy = model.evaluate(x_test, y_test)
#print('loss: ', loss, '\naccuracy: ', accuracy)

# 48934338

regression problem: Model doesnt learn, learns a horizental line!

SoF answer:  sgd = SGD(lr=0.001) and solves the issue.

But HomaTect fixes this by input standarization.

Also Acuuracy not a good metric

In [ ]:
from keras.models import Sequential #Added
from keras.layers import Dense # Added
import matplotlib.pyplot as plt # Added
from keras.optimizers import SGD # Added
import numpy as np # Added
from sklearn.preprocessing import StandardScaler # Added
import time

X = np.arange(0.0, 5.0, 0.1, dtype='float32').reshape(-1,1)
y = 5 * np.power(X,2) + np.power(np.random.randn(50).reshape(-1,1),3)

# Preproccessing Added
sc_x = StandardScaler()
sc_y = StandardScaler()
X = sc_x.fit_transform(X)
y = y.reshape(-1,1)
y = sc_y.fit_transform(y)



# ---------------------------- Added To check NORMALIZATION
#print("shape: ", X.shape)
#print("mean After normalization: ", numpy.mean(X, axis = 0))
#print("var After normalization: ", numpy.var(X, axis = 0))
#print("min After normalization: ", numpy.min(X, axis = 0))
#print("max After normalization: ", numpy.max(X, axis = 0))


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb



model = Sequential()
model.add(Dense(50, activation='relu', input_dim=1))
model.add(Dense(30, activation='relu', kernel_initializer='random_uniform')) # Was init='uniform'
model.add(Dense(1, activation='linear')) # Was model.add(Dense(output_dim=1, activation='linear'))
sgd = SGD(0.1); # Was sgd = SGD(lr=0.1);
model.compile(loss='mse', optimizer=sgd, metrics=['accuracy'] )
model.fit(X, y, epochs=1000) # Was nb_epoch=1000


modeltype = "Regression" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "[-n, n]" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 1 # Number of classes for classification and target shape for regression
input_shape = (1,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")


# --------------- Model is not trained
#predictions = model.predict(X) # No reverse!
#plt.scatter(X, y,edgecolors='g')
#plt.plot(X, predictions,'r')
#plt.legend(['Actual Y', 'Predicted Y'])
#plt.show()

 # 49117607
Multiclass calssification with mnist  
when filters are (5, 5) instead of (3, 3), Crashes with error: ValueError: Computed output size would be negative. Received `inputs shape=(None, 4, 4, 64)`, `kernel shape=(6, 6, 64, 64)`, `dilation_rate=[1 1].  
reproduction of error was posibble with (18, 18)  
size = ((input_size - kernel_size) / stride) + 1  
Decrease filter size  



In [ ]:
import keras
import numpy as np
import pandas as pd
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten, Conv2D, MaxPooling2D, BatchNormalization
from keras.optimizers import Adam
import time


(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.reshape(X_train.shape[0], 28, 28, 1)
X_test = X_test.reshape(X_test.shape[0], 28, 28, 1)

X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

X_train/=255
X_test/=255
number_of_classes = 10

Y_train = keras.utils.to_categorical(y_train, number_of_classes)
Y_test = keras.utils.to_categorical(y_test, number_of_classes)
input_shape = (28,28,1) # Added



start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb



model = Sequential()
model.add(Conv2D(32, (5, 5), padding='same', data_format='channels_last', input_shape=input_shape))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv2D(32, (18, 18)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))
model.add(Conv2D(64, (18, 18), padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv2D(64, (18, 18))) # Was model.add(Conv2D(64, (6, 6)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(512))
model.add(Activation('relu'))
model.add(Dropout(0.25))
model.add(Dense(10))
model.add(Activation('softmax'))

model.compile(loss='categorical_crossentropy', optimizer=Adam(), metrics=['accuracy'])
# fit generator removed
# gen = ImageDataGenerator(rotation_range=8, width_shift_range=0.08, shear_range=0.3, height_shift_range=0.08, zoom_range=0.08)
# test_gen = ImageDataGenerator()
# train_generator = gen.flow(X_train, Y_train, batch_size=64)
# test_generator = test_gen.flow(X_test, Y_test, batch_size=64)
# model.fit_generator(train_generator, steps_per_epoch=60000//64, epochs=5, validation_data=test_generator, validation_steps=10000//64)



modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 10 # Number of classes for classification and target shape for regression
input_shape = (28,28) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 50079585
Multi class classification with Car Damage severity detection

Low accuracy around 30%


Number of output nodes, Activation and Loss

Also specify class_mode='categorical' in data generators

In [ ]:
# from keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D
from keras.layers import Activation, Dropout, Flatten, Dense
from keras import backend as K
import time

img_width, img_height = 150, 150
# train_data_dir = 'dataset/data3a/training'
# validation_data_dir = 'dataset/data3a/validation'
nb_train_samples = 979
nb_validation_samples = 171
epochs = 10
batch_size = 16

if K.image_data_format() == 'channels_first':
    input_shape = (3, img_width, img_height)
else:
    input_shape = (img_width, img_height, 3)



start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb



model = Sequential()
model.add(Conv2D(32, (3, 3), input_shape=input_shape))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(32, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(64, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(64))
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(1))
# + model.add(Dense(3))
model.add(Activation('sigmoid'))
# + model.add(Activation('softmax'))
model.compile(loss='binary_crossentropy', optimizer='rmsprop', metrics=['accuracy'])
# + model.compile(loss='categorical_crossentropy', optimizer='rmsprop', metrics=['accuracy'])

# train_datagen = ImageDataGenerator( rescale=1. / 255, shear_range=0.2, zoom_range=0.2, horizontal_flip=True)
# test_datagen = ImageDataGenerator(rescale=1. / 255)
# train_generator = train_datagen.flow_from_directory( train_data_dir, target_size=(img_width, img_height), batch_size=batch_size, class_mode='binary')
# validation_generator = test_datagen.flow_from_directory( validation_data_dir, target_size=(img_width, img_height), batch_size=batch_size, class_mode='binary')
# model.fit_generator( train_generator, steps_per_epoch=nb_train_samples // batch_size, epochs=epochs, validation_data=validation_generator, validation_steps=nb_validation_samples // batch_size)
# model.save_weights('first_try.h5')



modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 3 # Number of classes for classification and target shape for regression
input_shape = (150,150) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 50306988

Binary classification

Accuracy 50%, predicts all in either class

Activation and loss, number of output nodes  

Normalize, Increase epoch

Preproccessing Added from SoF suggestion

In [ ]:
import random
import numpy as np
from keras.models import Sequential
from keras.layers import Dense
from sklearn.preprocessing import StandardScaler # Added
import time


# Preproccessing Added
n_samples = 200
n_feats = 2
n_classes = 2
cls0 = np.random.uniform(low=0.2, high=0.4, size=(n_samples,n_feats))
cls1 = np.random.uniform(low=0.5, high=0.7, size=(n_samples,n_feats))
x_train = np.concatenate((cls0, cls1))
y_train = np.concatenate((np.zeros((n_samples,)), np.ones((n_samples,))))
indices = np.arange(x_train.shape[0])
np.random.shuffle(indices)
x_train = x_train[indices]
y_train = y_train[indices]


scaler = StandardScaler() # Added
x_train = scaler.fit_transform(x_train) # Added


# --------------- Added To check NORMALIZATION
#print("shape: ", x_train.shape)
#print("mean After normalization: ", numpy.mean(x_train, axis = 0))
#print("var After normalization: ", numpy.var(x_train, axis = 0))
#print("min After normalization: ", numpy.min(x_train, axis = 0))
#print("max After normalization: ", numpy.max(x_train, axis = 0))



start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = Sequential()
model.add(Dense(units=2, activation='sigmoid', input_shape= (n_feats,)))
model.add(Dense(units=n_classes, activation='softmax'))
# + model.add(Dense(units=1, activation='sigmoid'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
# + model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=40, batch_size=32, verbose=True) # epochs were 5


modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (2,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 50426349


Multi output regression: output is colored image, deep colorization model using the CIFAR dataset

X_train dimensions = [50000,32,32] and Y_train = [50000,32,32,2]

Crashes with error: Input 0 is incompatible with layer conv2d_121: expected ndim=4, found ndim=5

My error: Kernel shape must have the same length as input, but received kernel of shape (3, 3, 1, 64) and input of shape (None, 50000, 32, 32, 1).

remove 50000 from input shape. batch size is specified in model.fit(). In this case it would become (128,50000,32,32,1).

NOT A CLASSIFICATION/ REGRESSION problem!

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Activation, Dropout, Flatten, Dense
import time

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = Sequential()
model.add(Conv2D(64, (3, 3), activation='relu', padding='same',input_shape = (50000,32,32,1)))
# + model.add(Conv2D(64, (3, 3), activation='relu', padding='same',input_shape = (32,32,1)))
model.add(Conv2D(64, (3, 3), activation='relu', padding='same', strides=2))
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(512, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(256, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(2, (3, 3), activation='tanh', padding='same'))
model.compile(optimizer='rmsprop', loss='mse')
# model_info = model.fit(X_train, Y_train, batch_size=128, epochs=200)


modeltype = "Regression" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "[-1, 1]" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = (32,32,2) # Number of classes for classification and target shape for regression
input_shape = (32,32) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 50481178

Accuracy always zero

Datset: Open, High, Low, Close stock data and output learning signal as -1 and +1

DeepLocalizes fix is irrelevant

outpout needs to be 0/1 not -1/1
and more epochs

SoF log after changing the output increases from zero to 50%. It probably needs more epochs  
Normalization makes convergence faster  

Correct code from DeepLocalize

In [ ]:
# Added Imports
from keras.preprocessing.sequence import pad_sequences
from sklearn.datasets import make_multilabel_classification
from keras import Sequential
from keras.layers import Dense, Flatten, Masking, LSTM, GRU, Conv1D, Dropout, MaxPooling1D
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from keras.utils import to_categorical
from time import *
import keras
import sys
import time

# Preprocessing added from DeepLocalize
n = 1000         # Number of instances equals 1000
input_dim = 4            # Number of features
num_classes = 1  # Number of output classes

dummyX, dummyY = make_multilabel_classification(n_samples=n, n_features=input_dim, n_classes=num_classes)
labelEncoder = LabelEncoder()
dummyY = labelEncoder.fit_transform(dummyY)
x_train, x_test, y_train, y_test = train_test_split(dummyX, dummyY, test_size=0.20)
input_shape = (input_dim,)
# ----------------- start_time = time()

# ---------------- print ("X: ", x_train)
# ----------------- print("Y: ", y_train)


scaler = StandardScaler() # Added
x_train = scaler.fit_transform(x_train) # Added


#---------------------- Added To check NORMALIZATION
#print("shape: ", x_train.shape)
#print("mean After normalization: ", np.mean(x_train, axis = 0))
#print("var After normalization: ", np.var(x_train, axis = 0))
#print("min After normalization: ", np.min(x_train, axis = 0))
#print("max After normalization: ", np.max(x_train, axis = 0))



start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


layers = [10,20,30,40,50]
model = keras.models.Sequential()
model.add(keras.layers.Dense(layers[0], input_dim = input_dim, activation='relu'))
for layer in layers[1:]:
    model.add(keras.layers.Dense(layer, activation='relu'))
model.add(keras.layers.Dense(1, activation='sigmoid')) # Was model.add(keras.layers.Dense(1, activation='sigmoid'))
model.compile(loss = 'binary_crossentropy', optimizer = 'adam', metrics = ['accuracy']) # Was model.compile(loss = 'binary_crossentropy', optimizer = 'adam', metrics = ['accuracy'])
model.fit(x_train, y_train, validation_split = 0.10, epochs = 50, batch_size = 10, shuffle = True, verbose = 2)
# ------------------------- end_time = time()


modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (4,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")


# --------------- Model is not trained yet
#loss, accuracy = model.evaluate(x_test, y_test)
#print("\nLoss: %.2f, Accuracy: %.2f%%" % (loss, accuracy*100))
#predt = model.predict(x_test)
#print(predt)
#print("time", (end_time - start_time))
#sys.exit(1)

# 50555434

Multiclass classification

Model is not learning, wrong output

The output vector should be of probabilty rates which are between 0 and one and to sum to 1

Commented out dataset related lines

Activation and loss


In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from keras import optimizers
import numpy
import time

# X = numpy.loadtxt("compiledFeatures.csv", delimiter=",")
# Y = numpy.loadtxt("naive_compiledDate.csv", delimiter=",")


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb
input_shape=(6,) # ADDED for HomaTect

model = Sequential()
model.add(Dense(20, kernel_initializer='random_uniform', activation='relu', input_shape=input_shape)) # input_shape ADDED
model.add(Dense(15, kernel_initializer='random_uniform', activation='relu'))# Was model.add(Dense(15, init='normal', activation='relu'))
model.add(Dense(6, kernel_initializer='random_uniform', activation='relu')) # Was model.add(Dense(6, init='normal', activation='relu'))
model.add(Dense(6, kernel_initializer='random_uniform', activation='sigmoid')) # Was model.add(Dense(6, kernel_initializer='random_uniform', activation='softmax'))
# + model.add(Dense(6, kernel_initializer='random_uniform', activation='softmax'))
model.compile(optimizer = "adam", loss = 'mae')
# + model.compile(optimizer = "adam", loss = 'categorical_crossentropy')
# model.fit(X, Y, epochs=2000,  verbose=2, validation_split = 0.15)
# predictions = model.predict(X)



modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 6 # Number of classes for classification and target shape for regression
input_shape = (6,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 51181393

Linear regression with incorrect result

SoF suggestion was changing the lr to 0.1 and it works: optimizers.RMSprop(lr=0.1)optimizers.

small dataset so increase epochs

Data normalization solved the problem even with 200 epochs

In [ ]:
from pylab import *
from keras.models import Sequential
from keras.layers import Dense
from sklearn.preprocessing import StandardScaler # Added To normalize
import time

data = data = linspace(1,2,100).reshape(-1,1)
y = data*5


print(data.shape)

# Added to standardize
X_train = data
y_train = y
sc_x = StandardScaler() # Added to normalize
sc_y = StandardScaler() # Added to normalize
X_train = sc_x.fit_transform(data) # Added to normalize
y_train = y.reshape(-1,1)
y_train = sc_y.fit_transform(y) # Added to normalize


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb
input_shape = (1,) # Added for HomaTect

def baseline_model():
    model = Sequential()
    model.add(Dense(1, activation = 'linear', input_shape=input_shape)) # input_shape ADDED
    model.compile(optimizer = 'rmsprop', loss = 'mean_squared_error', metrics = ['accuracy'])
    # + model.compile(optimizer = 'rmsprop', loss = 'mean_squared_error')
    return model   
regr = baseline_model()


modeltype = "Regression" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "[-n, n]" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 1 # Number of classes for classification and target shape for regression
input_shape = (1,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")


regr.fit(X_train,y_train,epochs =600,batch_size = 32)
plot(X_train, regr.predict(X_train), 'r', X_train,y_train, 'k.')

# 51749207
Binary classification

Accuracy around 30%

Change activation

Commented out dataset related lines

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Conv1D, MaxPooling1D, Dropout, Flatten
import time


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = Sequential()
model.add(Conv1D(filters=20, kernel_size=4,activation='relu',padding='same',input_shape=(600,1)))
model.add(MaxPooling1D(pool_size = 2))
model.add(Dropout(0.3))
model.add(Flatten())
model.add(Dense(50, activation='relu', input_dim = 600))
model.add(Dense(1, activation='softmax')) 
# + model.add(Dense(1, activation='sigmoid'))

model.compile(loss="binary_crossentropy", optimizer="nadam", metrics=['accuracy'])

# model.fit(np.array(X), np.array(Y), epochs = 100, batch_size=8, verbose=1, validation_data=(np.array(X1),np.array(Y1)))
# scores = model.evaluate(np.array(X1), np.array(Y1), verbose=0)



modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (600,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 51930566

Multiclass classification: iris dataset. Input of four dimensions.

Change activation to softmax

Dataset related lines commented out

In [ ]:
from keras.models import Sequential # Added
from keras.layers import Dense # Added
import time


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = Sequential()
model.add(Dense(4, input_dim=4, kernel_initializer='random_uniform', activation='relu')) # Was model.add(Dense(4, input_dim=4, init='normal', activation='relu'))
model.add(Dense(3, kernel_initializer='random_uniform', activation='sigmoid')) # Was model.add(Dense(3, init='normal', activation='sigmoid'))
# + model.add(Dense(3, kernel_initializer='random_uniform', activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
# model.fit(X_train, Y_train,batch_size=5, epochs=50, verbose=1, validation_data=(X_test, Y_test))
#score = model.evaluate(X_test, Y_test, verbose=0)



modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", [0, n], [-n, n]] for regression
output_shape = 3 # Number of classes for classification and target shape for regression
input_shape = (4,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 52800582

Regression

Inaccurate results

Standardize data

In [ ]:
import numpy as np
from keras.layers import Dense, Activation
from keras.models import Sequential
import matplotlib.pyplot as plt
import math
from sklearn.preprocessing import StandardScaler # Added To normalize
import time

x = np.arange(-100, 100, 0.5)
y = x**4

# Standardization from DeepLocalize Added
x_scaler = StandardScaler()
y_scaler = StandardScaler()
x = x_scaler.fit_transform(x[:, None])
y = y_scaler.fit_transform(y[:, None])


# ------------------------ Added To check NORMALIZATION
#print("shape: ", x.shape)
#print("mean After normalization: ", np.mean(x, axis = 0))
#print("var After normalization: ", np.var(x, axis = 0))
#print("min After normalization: ", np.min(x, axis = 0))
#print("max After normalization: ", np.max(x, axis = 0))




start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb



model = Sequential()
model.add(Dense(50, input_shape=(1,)))
model.add(Activation('sigmoid')) 
# + model.add(Activation('relu'))
model.add(Dense(50) )
model.add(Activation('elu'))
# + model.add(Activation('relu'))
model.add(Dense(1))
model.compile(loss='mse', optimizer='adam')



modeltype = "Regression" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "[-n, n]" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", "[0, n]", "[-n, n]"] for regression
output_shape = 1 # Number of classes for classification and target shape for regression
input_shape = (1,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")



# -------------------------- Model is not trained yet

#t1 = time.process_time() # Was t1 = time.clock()
#for i in range(1): # Was 100 bit took too long and seems unneccessary
#    model.fit(x, y, epochs=500, batch_size=len(x), verbose=0) # epochs were 1000
#    predictions = model.predict(x) # Didnt reverse fit?
#    #print (i," ", np.mean(np.square(predictions - y))," t: ", process_time()-t1)
#    # plt.hold(False)
#    plt.plot(x, y, 'b', x, predictions, 'r--')
#    #plt.hold(True)
#    plt.ylabel('Y / Predicted Value')
#    plt.xlabel('X Value')
#    #plt.title([str(i)," Loss: ",np.mean(np.square(predictions - y))," t: ", str(process_time()-t1)])
#    plt.pause(0.001)

#plt.show()

# 53119432

Binary Classification

Crashes with error

Add Flatten layer before dense

commented out datset related lines

In [ ]:
from keras.layers import Dense, Conv1D, MaxPooling1D, Dropout, Flatten
from keras.models import Sequential
import time


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model=Sequential()
# inputBatch = inputBatch.reshape(24,30, 1)
model.add(Conv1D(64, 3, activation='relu', input_shape=(30, 1)))
model.add(Conv1D(64, 3, activation='relu'))
model.add(MaxPooling1D(pool_size=4,strides=None, padding='valid'))
model.add(Conv1D(128, 3, activation='relu'))
model.add(Conv1D(128, 3, activation='relu'))
model.add(Dropout(0.5))
# + model.add(Flatten())
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
# model.fit(inputBatch,ponlabel,batch_size=24,epochs=20,validation_data=(inputBatch, ponlabel))




modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", "[0, n]", "[-n, n]"] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (30,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 55328966

Multi class classification with MNIST

used original MNIST because smaller MNIST dataset used here is not available

Loss nan, accuracy too low

output activation and data normalization

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import time

# !git clone https://github.com/DanorRon/data
# %cd data
# !ls

batch_size = 32
epochs = 10
alpha = 0.0001
lambda_ = 0
h1 = 50

train = pd.read_csv('./Datasets/MNIST/mnist_train.csv')
test = pd.read_csv('./Datasets/MNIST/mnist_test.csv')


train = train.loc['1':'5000', :]
test = test.loc['1':'2000', :]
train = train.sample(frac=1).reset_index(drop=True)
test = test.sample(frac=1).reset_index(drop=True)
x_train = train.loc[:, '1x1':'28x28']
y_train = train.loc[:, 'label']
x_test = test.loc[:, '1x1':'28x28']
y_test = test.loc[:, 'label']
x_train = x_train.values
y_train = y_train.values
x_test = x_test.values
y_test = y_test.values

# Added to normalize
x_train = x_train/255 # x_train = x_train.values/255
y_train = y_train # x_train = x_train.values/255
x_test = x_test/255 # x_train = x_train.values/255
y_test = y_test # x_train = x_train.values/255

nb_classes = 10
targets = y_train.reshape(-1)
y_train_onehot = np.eye(nb_classes)[targets]

nb_classes = 10
targets = y_test.reshape(-1)
y_test_onehot = np.eye(nb_classes)[targets]



start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb



model = tf.keras.Sequential()
model.add(layers.Dense(784, input_shape=(784,)))
model.add(layers.Dense(h1, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(lambda_)))
model.add(layers.Dense(10, activation='sigmoid', kernel_regularizer=tf.keras.regularizers.l2(lambda_)))
# + model.add(layers.Dense(10, activation='softmax', kernel_regularizer=tf.keras.regularizers.l2(lambda_)))
model.compile(optimizer=tf.optimizers.SGD(alpha), loss = 'categorical_crossentropy', metrics = ['accuracy']) # Was optimizer=tf.train.GradientDescentOptimizer(alpha) and change because of tensorlow version


modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", "[0, n]", "[-n, n]"] for regression
output_shape = 10 # Number of classes for classification and target shape for regression
input_shape = (784,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")


print(np.mean(x_train))
print(np.var(x_train))

model.fit(x_train, y_train_onehot, epochs=epochs, batch_size=batch_size)

# 55731589

Crashes with error:  ``ValueError: Negative dimension size caused by subtracting 3 from 2 for 
  'conv2d_24/convolution' (op: 'Conv2D') with input shapes: [?,2,2,128], 
  [3,3,128,128].

but I get no error!

SoF Explanation: Default layer padding is valid, which means no padding. This way dimension reduce from 32 not to 16 but to 15. You can use padding='same' instead. In this case output has the same length as the original input.



In [ ]:
from keras import layers
from keras import models
import time


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb



model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))  # SoF suggested adding padding='same'
model.add(layers.MaxPooling2D((2, 2))) # SoF suggested adding padding='same'
model.add(layers.Conv2D(64, (3, 3), activation='relu')) # SoF suggested adding padding='same'
model.add(layers.MaxPooling2D((2, 2))) # SoF suggested adding padding='same'
model.add(layers.Conv2D(128, (3, 3), activation='relu')) # SoF suggested adding padding='same'
model.add(layers.MaxPooling2D((2, 2))) # SoF suggested adding padding='same'
model.add(layers.Conv2D(128, (3, 3), activation='relu')) # SoF suggested adding padding='same'
model.add(layers.MaxPooling2D((2, 2))) # SoF suggested adding padding='same'
model.add(layers.Flatten())
model.add(layers.Dense(512, activation='relu'))
model.add(layers.Dense(1, activation='sigmoid'))




modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", "[0, n]", "[-n, n]"] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (32,32) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 55776436

Multi class classification to predict one's emotion from the physical signals

onehot encoded

accuracy is not going more than 0.2 and it is fluctuating up and down

Commented out dataset related lines

Neuralint results are irrelevant

Dont know what is wrong

In [ ]:
import keras
from keras.layers import Conv1D, Dropout, BatchNormalization, Activation, MaxPooling1D, Flatten, Dense
from keras. models import Sequential
import time


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = Sequential()
model.add(Conv1D(100,700,activation='relu',input_shape=(18000,1))) #kernel_size is 700 because 18000 rows = 60 seconds so 700 rows = ~2.33 seconds and there is two heart beat peak in every 2 second for ecg signal.
model.add(Conv1D(50,700))
model.add(Dropout(0.5))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling1D(4))
model.add(Flatten())
model.add(Dense(6,activation='softmax'))

adam = keras.optimizers.Adam(0.0001, beta_1=0.9, beta_2=0.999, epsilon=None, decay=0.0, amsgrad=False) # Was lr = 0.0001

model.compile(optimizer = adam, loss = 'categorical_crossentropy', metrics = ['acc'])
# model.fit(train_x,train_y,epochs = 50, batch_size = 32, validation_split=0.33, shuffle=False)



modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", "[0, n]", "[-n, n]"] for regression
output_shape = 6 # Number of classes for classification and target shape for regression
input_shape = (18000,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 56380303
Multiclass classification: takes a number in binary form, and outputs its modulo when dividing by 7.

the accuracy is around 1/7, which is the accuracy you'd expect from a completely random algorithm

Labels seem to be corrupt and make no sense.

SoF finally suggests RNN or adding and embedding layer for such a problem.

HomaTect cant make the model work but it will make useful suggestions: Number of output neurons and onehot encoding.

In [ ]:
import keras.models
import numpy as np
from python_toolbox import random_tools
import time

RADIX = 7

def _get_number(vector):
    return sum(x * 2 ** i for i, x in enumerate(vector))

def _get_mod_result(vector):
    return _get_number(vector) % RADIX

def _number_to_vector(number):
    binary_string = bin(number)[2:]
    if len(binary_string) > 20:
        raise NotImplementedError
    bits = (((0,) * (20 - len(binary_string))) +
            tuple(map(int, binary_string)))[::-1]
    assert len(bits) == 20
    return np.c_[bits]


def get_mod_result_vector(vector):
    return _number_to_vector(_get_mod_result(vector))


def main():


    start_time = time.perf_counter()
    %run kerassource/HomaTect.ipynb
    
    
    model = keras.models.Sequential(
        (
            keras.layers.Dense(
                units=20, activation='relu', input_dim=20
            ),
            keras.layers.Dense(
                units=20, activation='relu'
            ),
            keras.layers.Dense(
                units=20, activation='softmax'
            )
        )
    )
    model.compile(optimizer='sgd',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
    target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", "[0, n]", "[-n, n]"] for regression
    output_shape = 7 # Number of classes for classification and target shape for regression
    input_shape = (20,) # Just input shape without batch size or channels
    HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
    end_time = time.perf_counter()
    print(f"It took {end_time - start_time} seconds")

    
    

    data = np.random.randint(2, size=(10000, 20))  
    labels = np.vstack(list(map(get_mod_result_vector, data)))
    
    print("@", data[0])
    print(labels[0])
    print("@", data[1])
    print(labels[1])
    print("@", data[2])
    print(labels[2])
    print("@", data[3])
    print(labels[3])
    print("@", data[4])
    print(labels[4])
    print("@", data[5])
    print(labels[5])
    print("@", data[6])
    print(labels[6])
    print("@", data[7])
    print(labels[7])

    model.fit(data, labels, epochs=10, batch_size=50)
    
    def predict(number):
        foo = model.predict(_number_to_vector(number))
        return _get_number(tuple(map(round, foo[0])))
    def is_correct_for_number(x):
        return bool(predict(x) == x % RADIX)

    # ----------------------------- Model is not trained yet
    #predict(7)
    #sample = random_tools.shuffled(range(2 ** 20))[:500]
    #print('Total accuracy:')
    #print(sum(map(is_correct_for_number, sample)) / len(sample))
    #print(f'(Accuracy of random algorithm is {1/RADIX:.2f}')


if __name__ == '__main__':
    main()

# 58609115

Multiclass classification

Incorrect prediction (random data but labelling follows a logic)

Output activations, hiddel layer actiovations, loss

commented out dataset related lines because deeplocalizes dataset is incorrect

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import OneHotEncoder
from keras import Sequential
from keras.layers import Dense
from sklearn.model_selection import train_test_split
import time

# df = pd.read_csv("./Datasets/export78.csv")

# onehotencoder = OneHotEncoder(categorical_features = [1])
# data2 = onehotencoder.fit_transform(df).toarray()
# dataset = pd.DataFrame(data2)

# X= dataset.iloc[:,3557].astype(float)
# y= dataset.iloc[:,0:3557].astype(int)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


classifier = Sequential()
classifier.add(Dense(3557, activation='sigmoid', kernel_initializer='random_normal', input_dim=1))
# + classifier.add(Dense(3557, activation='relu', kernel_initializer='random_normal', input_dim=1))
classifier.add(Dense(3557, activation = 'sigmoid', kernel_initializer='random_normal'))
# + classifier.add(Dense(3557, activation = 'relu', kernel_initializer='random_normal'))
classifier.add(Dense(3557, activation = 'sigmoid', kernel_initializer='random_normal'))
# + classifier.add(Dense(3557, activation = 'softmax', kernel_initializer='random_normal'))
classifier.compile(optimizer ='adam',loss='binary_crossentropy', metrics=['accuracy'])
# + classifier.compile(optimizer ='adam',loss='categorical_crossentropy', metrics=['accuracy'])

# classifier.fit(X_train,y_train, batch_size=50, epochs=10)
# accr = classifier.evaluate(X_test, y_test)
# print('Test set\n  Loss: {:0.3f}\n  Accuracy: {:0.3f}'.format(accr[0] ,accr[1]))
# classifier.save("model.h67")
# data1 = np.array(X_test)
# List = [data1]
# model = tf.keras.models.load_model("model.h67")
# prediction = model.predict([(data1)])
# target = (np.argmax(prediction, axis=0))
# dataset1 = pd.DataFrame(target)
# display(dataset1)



modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", "[0, n]", "[-n, n]"] for regression
output_shape = 3557 # Number of classes for classification and target shape for regression
input_shape = (1,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 58844149
Binary classification with Zero accuracy

Commented out dataset related lines

Output activation

In [ ]:
from keras import Sequential
from keras.layers import Dropout, Activation, Conv2D, MaxPooling2D, Flatten
# from keras.preprocessing.image import ImageDataGenerator
import time


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = Sequential()
model.add(Conv2D(96, kernel_size=11, padding="same", input_shape=(300, 300, 1), activation = 'relu'))
model.add(MaxPooling2D(pool_size=(3, 3), strides=(2, 2)))
model.add(Conv2D(128, kernel_size=3, padding="same", activation = 'relu'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
model.add(Conv2D(128, kernel_size=3, padding="same", activation = 'relu'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
model.add(Flatten())
model.add(Dense(units= 300, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(1))
model.add(Activation("softmax")) 
# + model.add(Activation("sigmoid"))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# datagen = ImageDataGenerator(featurewise_center=True, rotation_range=90, fill_mode='nearest', validation_split = 0.2)
# datagen.fit(train)
# train_generator = datagen.flow(train, train_labels, batch_size=8)
# history = model.fit_generator(generator=train_generator, use_multiprocessing=True, steps_per_epoch = len(train_generator) / 8, epochs = 5, workers=20)



modeltype = "Binary Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", "[0, n]", "[-n, n]"] for regression
output_shape = 2 # Number of classes for classification and target shape for regression
input_shape = (300,300) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 59278771

Multiclass classification

Very low accuracy (around 32%)

standardize data, Output activation, increase number of neurons - accuracy become 90%

removed KerasClassifier cause it's not supported

In [ ]:
import numpy 
import pandas 
from keras.models import Sequential 
from keras.layers import Dense 
#from keras.wrappers.scikit_learn import KerasClassifier 
#from keras.utils import np_utils 
#from sklearn.model_selection import cross_val_score 
#from sklearn.model_selection import KFold 
#from sklearn.preprocessing import LabelEncoder 
#from sklearn.pipeline import Pipeline
from sklearn.datasets import load_iris
from sklearn.preprocessing import MinMaxScaler #Added
import keras
import time


seed = 7 
numpy.random.seed(seed) 
X, encoded_Y = load_iris(return_X_y=True)
#dataframe = pandas.read_csv("/content/drive/My Drive/iris.data", header=None)
#dataset = dataframe.values 
#X = dataset[:,0:4].astype(float) 
#Y = dataset[:,4] 

mms = MinMaxScaler()
X = mms.fit_transform(X)
dummy_y = keras.utils.to_categorical(encoded_Y)
#encoder = LabelEncoder() 
#encoder.fit(Y) 
#encoded_Y = encoder.transform(Y) 
#dummy_y = np_utils.to_categorical(encoded_Y) 

def baseline_model():
    model = Sequential()
    model.add(Dense(4, input_dim=4, activation="relu", kernel_initializer="normal"))
    # + model.add(Dense(8, activation="relu", kernel_initializer="normal")) # DeepLocalize Added
    model.add(Dense(3, activation="sigmoid", kernel_initializer="normal"))
    # + model.add(Dense(3, activation="softmax", kernel_initializer="normal"))
    model.compile(loss= 'categorical_crossentropy' , optimizer= 'adam' , metrics=[ 'accuracy' ])
    return model 
#estimator = KerasClassifier(build_fn=baseline_model, nb_epoch=200, batch_size=5, verbose=0) 
#kfold = KFold(n_splits=10, shuffle=True, random_state=seed) 
#results = cross_val_score(estimator, X, dummy_y, cv=kfold) 
#print("Accuracy: %.2f%% (%.2f%%)" % (results.mean()*100, results.std()*100))


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb

baseline_model()

modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", "[0, n]", "[-n, n]"] for regression
output_shape = 3 # Number of classes for classification and target shape for regression
input_shape = (4,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 59325381
Multiclass classification

Low accuracy (bellow 10%)

lables are class numbers, should be transformed to onhot, loss, Scaling problem.

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import time

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
print(y_train[12])
print(np.shape(x_train))
print(np.shape(x_test))
x_train = x_train/255
x_test = x_test/255 # Was y_train = y_train/255

x_train = x_train.reshape(60000,28,28,1)
x_test = x_test.reshape(10000,28,28,1)
y_train = y_train.reshape(60000,1)
y_test = y_test.reshape(10000,1)


# Added
from tensorflow.keras.utils import to_categorical
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print(y_test[0])
print(y_test[1])

start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = keras.Sequential([
keras.layers.Conv2D(64,(3,3),(1,1),padding = "same",input_shape=(28,28,1)),
keras.layers.MaxPooling2D(pool_size = (2,2),padding = "valid"),
keras.layers.Conv2D(32,(3,3),(1,1),padding = "same"),
keras.layers.MaxPooling2D(pool_size = (2,2),padding = "valid"),
keras.layers.Flatten(),
keras.layers.Dense(128,activation = "relu"),
keras.layers.Dense(10,activation = "softmax")])

model.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics  = ['accuracy'])

model.fit(x_train,y_train,epochs=10)

# --------------- Model is not trained yet
#test_loss,test_acc = model.evaluate(x_test,y_test)
#print("\ntest accuracy:",test_acc)


modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", "[0, n]", "[-n, n]"] for regression
output_shape = 10 # Number of classes for classification and target shape for regression
input_shape = (28,28) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 60566498

Multiclass classification

Commented out dataset related lines

SoF says it's an overfitting problem

Neuralint fixes irrelevant

In [ ]:
from keras import Sequential
from keras.layers import Dropout, Activation, Conv2D, MaxPooling2D, Flatten
import time

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1)


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = Sequential()
model.add(Conv2D(256, (3, 3), input_shape= (300, 300, 1), padding='same')) # Was input_shape=X.shape[1:] so gave a random input_shape to run
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=2))
model.add(Dropout(0.2))
model.add(Conv2D(256, (3, 3), padding='same'))
model.add(Dropout(0.2))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=2))
model.add(Conv2D(256, (3, 3), padding='same'))
model.add(Dropout(0.25))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=2))
model.add(Conv2D(256, (3, 3), padding='same'))
model.add(Dropout(0.25))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=2))
model.add(Flatten())
model.add(Dense(256))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.8))
model.add(Dense(3))
model.add(Activation('softmax'))

# tensorboard = TensorBoard(log_dir="CNN_Model_Rebuilt/logs/{}".format(NAME))
# augmented_checkpoint = ModelCheckpoint('CNN_Model_Rebuilt/best model/normalization-best.h5', monitor='val_loss', verbose=0, save_best_only=True, mode='auto')
# es = EarlyStopping(monitor='val_loss', min_delta=0, patience=20, verbose=0, mode='auto')

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
# model.fit(X_train, to_categorical(y_train), batch_size=32, epochs=100, validation_data=(X_test, to_categorical(y_test)), callbacks=[augmented_checkpoint, tensorboard, es], verbose=2)



modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Onehot Encoding" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", "[0, n]", "[-n, n]"] for regression
output_shape = 3 # Number of classes for classification and target shape for regression
input_shape = (300,300) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")

# 61030068

Multiclass classification
training_labels is (3560,) but there are 28 classe so it's not onehotencoded.  
class number  

Crashes with error: Error when checking target: expected dense_31 to have 3 dimensions, but got array with shape (3560, 1)

Add Faltten
Sparse categorical crossentropy

Commented out dataset related lines

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dense, MaxPooling1D, Flatten
from tensorflow.keras.optimizers import Adam
import time

# training_features = np.load('features.npy')
# training_labels = np.load('labels.npy')
# training_features = training_features.reshape(-1, 72, 1)


start_time = time.perf_counter()
%run kerassource/HomaTect.ipynb


model = Sequential()
model.add(Conv1D(64, 3, activation='relu', input_shape=(72, 1)))
model.add(MaxPooling1D(2))
model.add(Conv1D(64, 3, activation='relu'))
model.add(MaxPooling1D(2))
# + model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dense(28, activation='softmax'))

model.compile(Adam(0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
# model.fit(training_features, training_labels, batch_size=32, epochs=3, validation_split=0.1)



modeltype = "Multiclass Classification" # ["Binary Classification", "Multiclass Classification", "Multilabel Classification", "Regression"]
target = "Class Numbers" # ["Onehot Encoding", "Class Numbers"] for classification and ["[-1, 1]", "[0, 1]", "[-n, 0]", "[0, n]", "[-n, n]"] for regression
output_shape = 28 # Number of classes for classification and target shape for regression
input_shape = (72,) # Just input shape without batch size or channels
HomaTect(modeltype.casefold(), target.casefold().replace(" ", ""), input_shape, output_shape) # And for CNN, define input_shape in argument
end_time = time.perf_counter()
print(f"It took {end_time - start_time} seconds")